## Polymarket analytics playground (leaderboard + whale-move alerts)

This notebook gives you reusable functions around the docs you shared:

- `GET https://gamma-api.polymarket.com/events` (list events)
- `GET https://gamma-api.polymarket.com/events/{id}` (event detail)
- `GET https://data-api.polymarket.com/positions` (user positions)
- `GET https://data-api.polymarket.com/trades` (user/market trades)
- `GET https://data-api.polymarket.com/v1/leaderboard` (top traders)
- `GET https://bridge.polymarket.com/status/{address}` (bridge transaction status)
- `POST https://relayer-v2.polymarket.com/submit` (submit tx, auth required)

### Rate-limit notes (from docs)
- Gamma API general: ~4,000 / 10s
- Data API general: ~1,000 / 10s
- Data `/trades`: ~200 / 10s
- Data `/positions`: ~150 / 10s
- Gamma `/events`: ~500 / 10s
- Relayer `/submit`: ~25 / 1 min

Use batched polling and low-frequency loops for whale monitoring to avoid throttling.

In [1]:
import time
import hmac
import hashlib
import json
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import requests

GAMMA_BASE = "https://gamma-api.polymarket.com"
DATA_BASE = "https://data-api.polymarket.com"
BRIDGE_BASE = "https://bridge.polymarket.com"
RELAYER_BASE = "https://relayer-v2.polymarket.com"


class PolyClient:
    """Lightweight Polymarket HTTP client with basic retry/backoff."""

    def __init__(self, timeout: int = 20):
        self.timeout = timeout
        self.session = requests.Session()

    def _request(self, method: str, url: str, **kwargs) -> Any:
        max_retries = kwargs.pop("max_retries", 4)
        retry_sleep = kwargs.pop("retry_sleep", 0.6)

        for attempt in range(max_retries):
            try:
                resp = self.session.request(method, url, timeout=self.timeout, **kwargs)
                if resp.status_code in (429, 500, 502, 503, 504):
                    if attempt < max_retries - 1:
                        time.sleep(retry_sleep * (2 ** attempt))
                        continue
                resp.raise_for_status()
                return resp.json()
            except requests.RequestException:
                if attempt == max_retries - 1:
                    raise
                time.sleep(retry_sleep * (2 ** attempt))

    # ---------- Events ----------
    def list_events(self, **params) -> List[Dict[str, Any]]:
        url = f"{GAMMA_BASE}/events"
        data = self._request("GET", url, params=params)
        return data if isinstance(data, list) else []

    def get_event_by_id(self, event_id: int, include_chat: bool = False, include_template: bool = False) -> Dict[str, Any]:
        url = f"{GAMMA_BASE}/events/{event_id}"
        return self._request(
            "GET",
            url,
            params={"include_chat": include_chat, "include_template": include_template},
        )

    # ---------- Core data ----------
    def get_positions(
        self,
        user: str,
        limit: int = 100,
        offset: int = 0,
        sort_by: str = "TOKENS",
        sort_direction: str = "DESC",
        size_threshold: float = 1,
        **extra_params,
    ) -> List[Dict[str, Any]]:
        url = f"{DATA_BASE}/positions"
        params = {
            "user": user,
            "limit": limit,
            "offset": offset,
            "sortBy": sort_by,
            "sortDirection": sort_direction,
            "sizeThreshold": size_threshold,
            **extra_params,
        }
        data = self._request("GET", url, params=params)
        return data if isinstance(data, list) else []

    def get_trades(
        self,
        user: Optional[str] = None,
        market: Optional[str] = None,
        event_id: Optional[int] = None,
        side: Optional[str] = None,
        taker_only: bool = True,
        limit: int = 100,
        offset: int = 0,
    ) -> List[Dict[str, Any]]:
        url = f"{DATA_BASE}/trades"
        params: Dict[str, Any] = {
            "limit": limit,
            "offset": offset,
            "takerOnly": str(taker_only).lower(),
        }
        if user:
            params["user"] = user
        if market:
            params["market"] = market
        if event_id is not None:
            params["eventId"] = event_id
        if side:
            params["side"] = side

        data = self._request("GET", url, params=params)
        return data if isinstance(data, list) else []

    # ---------- Leaderboard ----------
    def get_leaderboard(
        self,
        category: str = "OVERALL",
        time_period: str = "DAY",
        order_by: str = "PNL",
        limit: int = 25,
        offset: int = 0,
        user: Optional[str] = None,
        user_name: Optional[str] = None,
    ) -> List[Dict[str, Any]]:
        url = f"{DATA_BASE}/v1/leaderboard"
        params: Dict[str, Any] = {
            "category": category,
            "timePeriod": time_period,
            "orderBy": order_by,
            "limit": limit,
            "offset": offset,
        }
        if user:
            params["user"] = user
        if user_name:
            params["userName"] = user_name

        data = self._request("GET", url, params=params)
        return data if isinstance(data, list) else []

    # ---------- Bridge ----------
    def get_bridge_transaction_status(self, address: str) -> Dict[str, Any]:
        url = f"{BRIDGE_BASE}/status/{address}"
        return self._request("GET", url)

    # ---------- Relayer ----------
    def submit_relayer_transaction(
        self,
        payload: Dict[str, Any],
        headers: Dict[str, str],
    ) -> Dict[str, Any]:
        """
        Docs: POST /submit (relayer-v2)
        Requires either Builder API key headers OR Relayer API key headers.
        """
        url = f"{RELAYER_BASE}/submit"
        return self._request("POST", url, json=payload, headers=headers)


client = PolyClient()
print("Client ready:", datetime.now(timezone.utc).isoformat())

Client ready: 2026-03-22T10:33:23.034519+00:00


In [2]:
def top_wallets(
    category: str = "OVERALL",
    time_period: str = "DAY",
    order_by: str = "PNL",
    limit: int = 10,
) -> List[str]:
    rows = client.get_leaderboard(
        category=category,
        time_period=time_period,
        order_by=order_by,
        limit=limit,
    )
    return [r.get("proxyWallet", "") for r in rows if r.get("proxyWallet")]


def trade_notional_usd(trade: Dict[str, Any]) -> float:
    size = float(trade.get("size", 0) or 0)
    price = float(trade.get("price", 0) or 0)
    return size * price


def detect_large_trades(
    trades: List[Dict[str, Any]],
    min_notional: float = 2_000,
) -> List[Dict[str, Any]]:
    alerts = []
    for t in trades:
        notional = trade_notional_usd(t)
        if notional >= min_notional:
            alerts.append(
                {
                    "proxyWallet": t.get("proxyWallet"),
                    "side": t.get("side"),
                    "title": t.get("title"),
                    "outcome": t.get("outcome"),
                    "price": t.get("price"),
                    "size": t.get("size"),
                    "notional": round(notional, 2),
                    "timestamp": t.get("timestamp"),
                    "transactionHash": t.get("transactionHash"),
                }
            )
    alerts.sort(key=lambda x: x["notional"], reverse=True)
    return alerts


def poll_wallets_for_big_moves(
    wallets: List[str],
    min_notional: float = 2_000,
    max_trades_each: int = 50,
) -> List[Dict[str, Any]]:
    all_alerts: List[Dict[str, Any]] = []
    for w in wallets:
        trades = client.get_trades(user=w, taker_only=True, limit=max_trades_each)
        wallet_alerts = detect_large_trades(trades, min_notional=min_notional)
        all_alerts.extend(wallet_alerts)
        time.sleep(0.12)  # soft pacing for rate-limit safety
    all_alerts.sort(key=lambda x: x["notional"], reverse=True)
    return all_alerts


def summarize_wallet_positions(wallet: str, top_n: int = 10) -> List[Dict[str, Any]]:
    positions = client.get_positions(user=wallet, limit=200, sort_by="CURRENT", sort_direction="DESC")
    cleaned = []
    for p in positions:
        cleaned.append(
            {
                "title": p.get("title"),
                "outcome": p.get("outcome"),
                "size": p.get("size"),
                "avgPrice": p.get("avgPrice"),
                "curPrice": p.get("curPrice"),
                "currentValue": p.get("currentValue"),
                "cashPnl": p.get("cashPnl"),
                "percentPnl": p.get("percentPnl"),
                "conditionId": p.get("conditionId"),
            }
        )
    cleaned.sort(key=lambda x: float(x.get("currentValue", 0) or 0), reverse=True)
    return cleaned[:top_n]

In [3]:
# --- Example workflow: fetch key players + detect big sudden moves ---

leaders = client.get_leaderboard(category="OVERALL", time_period="DAY", order_by="PNL", limit=5)
example_players = [
    {
        "rank": row.get("rank"),
        "wallet": row.get("proxyWallet"),
        "userName": row.get("userName"),
        "pnl": row.get("pnl"),
        "vol": row.get("vol"),
    }
    for row in leaders
]

print("Top example players:")
for p in example_players:
    print(p)

wallets = [p["wallet"] for p in example_players if p.get("wallet")]
alerts = poll_wallets_for_big_moves(wallets, min_notional=5_000, max_trades_each=100)

print("\nLarge-move alerts (sample):")
for a in alerts[:20]:
    print(a)

Top example players:
{'rank': '1', 'wallet': '0x492442eab586f242b53bda933fd5de859c8a3782', 'userName': '0x492442EaB586F242B53bDa933fD5dE859c8A3782-1766317541188', 'pnl': 484395.52294014953, 'vol': 562563.9181570001}
{'rank': '2', 'wallet': '0x03e8a544e97eeff5753bc1e90d46e5ef22af1697', 'userName': 'weflyhigh', 'pnl': 393426.5287023741, 'vol': 1203060.0250220003}
{'rank': '3', 'wallet': '0x2a2c53bd278c04da9962fcf96490e17f3dfb9bc1', 'userName': '0x2a2C53bD278c04DA9962Fcf96490E17F3DfB9Bc1-1772479215461', 'pnl': 358625.4104066574, 'vol': 5942738.443833998}
{'rank': '4', 'wallet': '0x6ac5bb06a9eb05641fd5e82640268b92f3ab4b6e', 'userName': '0p0jogggg', 'pnl': 322918.619174258, 'vol': 1091409.5851859997}
{'rank': '5', 'wallet': '0xead152b855effa6b5b5837f53b24c0756830c76a', 'userName': 'elkmonkey', 'pnl': 252018.8949802222, 'vol': 581254.241534}

Large-move alerts (sample):
{'proxyWallet': '0x03e8a544e97eeff5753bc1e90d46e5ef22af1697', 'side': 'BUY', 'title': 'Blue Jackets vs. Flyers: O/U 6.5', '

In [4]:
# --- Optional: event discovery helpers (from /events and /events/{id}) ---

def find_high_liquidity_events(min_liquidity: float = 100_000, limit: int = 50) -> List[Dict[str, Any]]:
    events = client.list_events(limit=limit, active=True, closed=False, ascending=False)
    rows = []
    for e in events:
        liq = float(e.get("liquidity") or 0)
        vol = float(e.get("volume") or 0)
        if liq >= min_liquidity:
            rows.append(
                {
                    "id": e.get("id"),
                    "title": e.get("title"),
                    "category": e.get("category"),
                    "liquidity": liq,
                    "volume": vol,
                    "endDate": e.get("endDate"),
                }
            )
    rows.sort(key=lambda x: x["liquidity"], reverse=True)
    return rows


def event_snapshot(event_id: int) -> Dict[str, Any]:
    e = client.get_event_by_id(event_id)
    markets = e.get("markets", []) or []
    return {
        "id": e.get("id"),
        "title": e.get("title"),
        "category": e.get("category"),
        "liquidity": e.get("liquidity"),
        "volume": e.get("volume"),
        "openInterest": e.get("openInterest"),
        "active": e.get("active"),
        "closed": e.get("closed"),
        "marketCount": len(markets),
    }


# quick samples:
high_liq = find_high_liquidity_events(min_liquidity=250_000, limit=100)
print(f"High-liquidity active events: {len(high_liq)}")
for row in high_liq[:10]:
    print(row)

High-liquidity active events: 27
{'id': '30615', 'title': '2026 FIFA World Cup Winner ', 'category': None, 'liquidity': 46087493.45219, 'volume': 348664179.35305846, 'endDate': '2026-07-20T00:00:00Z'}
{'id': '30829', 'title': 'Democratic Presidential Nominee 2028', 'category': None, 'liquidity': 44501505.51498, 'volume': 881549052.234146, 'endDate': '2028-11-07T00:00:00Z'}
{'id': '31875', 'title': 'Republican Presidential Nominee 2028', 'category': None, 'liquidity': 26773324.2888, 'volume': 455837629.798072, 'endDate': '2028-11-07T00:00:00Z'}
{'id': '31552', 'title': 'Presidential Election Winner 2028', 'category': None, 'liquidity': 25052929.889, 'volume': 437643369.00559735, 'endDate': '2028-11-07T00:00:00Z'}
{'id': '27830', 'title': '2026 NBA Champion', 'category': None, 'liquidity': 13766661.11454, 'volume': 252801645.0600708, 'endDate': '2026-07-01T00:00:00Z'}
{'id': '33506', 'title': 'UEFA Champions League Winner ', 'category': None, 'liquidity': 4438549.97362, 'volume': 2161514

In [5]:
# --- Advanced helpers: deduplicated alert stream + relayer signing scaffold ---

def stream_big_moves(
    wallets: List[str],
    min_notional: float = 10_000,
    poll_seconds: int = 20,
    loops: int = 6,
) -> List[Dict[str, Any]]:
    """
    Poll target wallets and alert only on NEW tx hashes over repeated loops.
    Useful for a simple whale-follow signal stream.
    """
    seen_hashes = set()
    emitted: List[Dict[str, Any]] = []

    for i in range(loops):
        batch_alerts = poll_wallets_for_big_moves(wallets, min_notional=min_notional, max_trades_each=100)
        new_items = []
        for a in batch_alerts:
            txh = (a.get("transactionHash") or "")
            key = txh if txh else json.dumps(a, sort_keys=True)
            if key in seen_hashes:
                continue
            seen_hashes.add(key)
            a["detectedAt"] = datetime.now(timezone.utc).isoformat()
            new_items.append(a)
            emitted.append(a)

        print(f"loop={i+1}/{loops} | new_alerts={len(new_items)}")
        for a in new_items[:20]:
            print(a)

        if i < loops - 1:
            time.sleep(poll_seconds)

    return emitted


def build_poly_builder_signature(secret: str, timestamp: str, passphrase: str) -> str:
    """
    HMAC-SHA256 signature helper for Builder API auth flow.
    Note: keep secrets in env vars, never hardcode.
    """
    message = f"{timestamp}{passphrase}".encode("utf-8")
    return hmac.new(secret.encode("utf-8"), message, hashlib.sha256).hexdigest()


def example_relayer_headers_from_env() -> Dict[str, str]:
    """
    Fill these from environment variables before calling /submit.
    """
    import os

    api_key = os.getenv("POLY_BUILDER_API_KEY", "")
    passphrase = os.getenv("POLY_BUILDER_PASSPHRASE", "")
    secret = os.getenv("POLY_BUILDER_SECRET", "")
    timestamp = str(int(time.time()))

    sig = build_poly_builder_signature(secret=secret, timestamp=timestamp, passphrase=passphrase)
    return {
        "POLY_BUILDER_API_KEY": api_key,
        "POLY_BUILDER_TIMESTAMP": timestamp,
        "POLY_BUILDER_PASSPHRASE": passphrase,
        "POLY_BUILDER_SIGNATURE": sig,
        "Content-Type": "application/json",
    }


# Example only (DO NOT run real tx unless payload is correct and reviewed):
# headers = example_relayer_headers_from_env()
# payload = {
#     "from": "0x...",
#     "to": "0x...",
#     "proxyWallet": "0x...",
#     "data": "0x...",
#     "nonce": "...",
#     "signature": "0x...",
#     "signatureParams": {
#         "gasPrice": "0",
#         "operation": "0",
#         "safeTxnGas": "0",
#         "baseGas": "0",
#         "gasToken": "0x0000000000000000000000000000000000000000",
#         "refundReceiver": "0x0000000000000000000000000000000000000000",
#     },
#     "type": "SAFE",
# }
# result = client.submit_relayer_transaction(payload, headers=headers)
# print(result)

## Alert persistence + notifications

This section adds:
- SQLite + CSV export for detected whale alerts
- Optional Discord webhook notifications
- Optional Telegram bot notifications

Set secrets in environment variables (do not hardcode):
- `DISCORD_WEBHOOK_URL`
- `TELEGRAM_BOT_TOKEN`
- `TELEGRAM_CHAT_ID`

In [6]:
import os
import csv
import sqlite3
from pathlib import Path


def init_alert_db(db_path: str = "logs/alerts.db") -> None:
    Path(db_path).parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS whale_alerts (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            detected_at TEXT,
            proxy_wallet TEXT,
            side TEXT,
            title TEXT,
            outcome TEXT,
            price REAL,
            size REAL,
            notional REAL,
            timestamp INTEGER,
            transaction_hash TEXT
        )
        """
    )
    cur.execute(
        """
        CREATE INDEX IF NOT EXISTS idx_whale_alerts_wallet_time
        ON whale_alerts(proxy_wallet, timestamp)
        """
    )
    conn.commit()
    conn.close()


def save_alerts_to_sqlite(alerts: List[Dict[str, Any]], db_path: str = "logs/alerts.db") -> int:
    init_alert_db(db_path)
    if not alerts:
        return 0

    rows = [
        (
            a.get("detectedAt"),
            a.get("proxyWallet"),
            a.get("side"),
            a.get("title"),
            a.get("outcome"),
            float(a.get("price", 0) or 0),
            float(a.get("size", 0) or 0),
            float(a.get("notional", 0) or 0),
            int(a.get("timestamp", 0) or 0),
            a.get("transactionHash"),
        )
        for a in alerts
    ]

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.executemany(
        """
        INSERT INTO whale_alerts(
            detected_at, proxy_wallet, side, title, outcome,
            price, size, notional, timestamp, transaction_hash
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        rows,
    )
    conn.commit()
    conn.close()
    return len(rows)


def save_alerts_to_csv(alerts: List[Dict[str, Any]], csv_path: str = "logs/alerts.csv") -> int:
    Path(csv_path).parent.mkdir(parents=True, exist_ok=True)
    fields = [
        "detectedAt", "proxyWallet", "side", "title", "outcome",
        "price", "size", "notional", "timestamp", "transactionHash"
    ]
    file_exists = Path(csv_path).exists()

    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        if not file_exists:
            writer.writeheader()
        for a in alerts:
            writer.writerow({k: a.get(k, "") for k in fields})

    return len(alerts)


def format_alert_message(alert: Dict[str, Any]) -> str:
    return (
        f"🚨 Whale move\n"
        f"wallet: {alert.get('proxyWallet')}\n"
        f"side: {alert.get('side')}\n"
        f"market: {alert.get('title')}\n"
        f"outcome: {alert.get('outcome')}\n"
        f"notional: ${float(alert.get('notional', 0) or 0):,.2f}\n"
        f"price: {alert.get('price')} | size: {alert.get('size')}\n"
        f"tx: {alert.get('transactionHash')}"
    )


def send_discord_alert(alert: Dict[str, Any], webhook_url: Optional[str] = None) -> bool:
    webhook = webhook_url or os.getenv("DISCORD_WEBHOOK_URL", "")
    if not webhook:
        return False

    payload = {"content": format_alert_message(alert)}
    r = requests.post(webhook, json=payload, timeout=15)
    return r.status_code in (200, 204)


def send_telegram_alert(alert: Dict[str, Any], bot_token: Optional[str] = None, chat_id: Optional[str] = None) -> bool:
    token = bot_token or os.getenv("TELEGRAM_BOT_TOKEN", "")
    chat = chat_id or os.getenv("TELEGRAM_CHAT_ID", "")
    if not token or not chat:
        return False

    url = f"https://api.telegram.org/bot{token}/sendMessage"
    payload = {"chat_id": chat, "text": format_alert_message(alert)}
    r = requests.post(url, json=payload, timeout=15)
    return r.status_code == 200


def persist_and_notify_alerts(
    alerts: List[Dict[str, Any]],
    db_path: str = "logs/alerts.db",
    csv_path: str = "logs/alerts.csv",
    notify_discord: bool = False,
    notify_telegram: bool = False,
    max_notify: int = 20,
) -> Dict[str, int]:
    written_db = save_alerts_to_sqlite(alerts, db_path=db_path)
    written_csv = save_alerts_to_csv(alerts, csv_path=csv_path)

    sent_discord = 0
    sent_telegram = 0

    for a in alerts[:max_notify]:
        if notify_discord and send_discord_alert(a):
            sent_discord += 1
        if notify_telegram and send_telegram_alert(a):
            sent_telegram += 1

    return {
        "saved_sqlite": written_db,
        "saved_csv": written_csv,
        "sent_discord": sent_discord,
        "sent_telegram": sent_telegram,
    }

In [7]:
# Example: persist latest detected alerts without sending notifications
# (set notify_discord=True / notify_telegram=True after env vars are configured)

if 'alerts' in globals() and isinstance(alerts, list):
    result = persist_and_notify_alerts(
        alerts,
        db_path="logs/alerts.db",
        csv_path="logs/alerts.csv",
        notify_discord=False,
        notify_telegram=False,
    )
    print("Persist result:", result)
else:
    print("No `alerts` list found yet. Run the big-move detection cell first.")

Persist result: {'saved_sqlite': 168, 'saved_csv': 168, 'sent_discord': 0, 'sent_telegram': 0}


## Continuous whale monitor loop (leaderboard-driven)

This loop refreshes top wallets from leaderboard, checks new large trades, deduplicates by tx hash, then persists + optional notifications.

In [8]:
def run_continuous_whale_monitor(
    loops: int = 6,
    poll_seconds: int = 30,
    leaderboard_limit: int = 8,
    min_notional: float = 10_000,
    category: str = "OVERALL",
    time_period: str = "DAY",
    order_by: str = "PNL",
    notify_discord: bool = False,
    notify_telegram: bool = False,
) -> Dict[str, Any]:
    """
    End-to-end loop:
    1) fetch top leaderboard wallets
    2) fetch latest trades for each wallet
    3) detect large notional moves
    4) deduplicate events across loops
    5) persist + optional notify
    """
    seen = set()
    totals = {
        "loops": loops,
        "alerts_detected": 0,
        "alerts_new": 0,
        "saved_sqlite": 0,
        "saved_csv": 0,
        "sent_discord": 0,
        "sent_telegram": 0,
    }

    for i in range(loops):
        leaders_now = client.get_leaderboard(
            category=category,
            time_period=time_period,
            order_by=order_by,
            limit=leaderboard_limit,
        )
        watch_wallets = [r.get("proxyWallet") for r in leaders_now if r.get("proxyWallet")]

        batch = poll_wallets_for_big_moves(
            wallets=watch_wallets,
            min_notional=min_notional,
            max_trades_each=100,
        )
        totals["alerts_detected"] += len(batch)

        new_alerts = []
        for a in batch:
            txh = (a.get("transactionHash") or "").strip()
            key = txh if txh else json.dumps(a, sort_keys=True)
            if key in seen:
                continue
            seen.add(key)
            a["detectedAt"] = datetime.now(timezone.utc).isoformat()
            a["leaderboardLoop"] = i + 1
            new_alerts.append(a)

        totals["alerts_new"] += len(new_alerts)

        if new_alerts:
            persist_result = persist_and_notify_alerts(
                new_alerts,
                db_path="logs/alerts.db",
                csv_path="logs/alerts.csv",
                notify_discord=notify_discord,
                notify_telegram=notify_telegram,
                max_notify=25,
            )
            totals["saved_sqlite"] += persist_result["saved_sqlite"]
            totals["saved_csv"] += persist_result["saved_csv"]
            totals["sent_discord"] += persist_result["sent_discord"]
            totals["sent_telegram"] += persist_result["sent_telegram"]

        print(
            f"loop {i+1}/{loops} | wallets={len(watch_wallets)} | "
            f"detected={len(batch)} | new={len(new_alerts)}"
        )

        if i < loops - 1:
            time.sleep(poll_seconds)

    return totals


# Safe sample run (short): no notifications
monitor_summary = run_continuous_whale_monitor(
    loops=2,
    poll_seconds=8,
    leaderboard_limit=5,
    min_notional=15_000,
    notify_discord=False,
    notify_telegram=False,
)
print("Monitor summary:", monitor_summary)

loop 1/2 | wallets=5 | detected=94 | new=94
loop 2/2 | wallets=5 | detected=94 | new=0
Monitor summary: {'loops': 2, 'alerts_detected': 188, 'alerts_new': 94, 'saved_sqlite': 94, 'saved_csv': 94, 'sent_discord': 0, 'sent_telegram': 0}


## 🌦️ Weather Market Intelligence

Five analytical lenses on Polymarket's weather/climate category:

| # | Idea | Module | What it does |
|---|------|--------|--------------|
| 1 | Weather Whale Monitor | `weather_whale_monitor.py` | Filter WEATHER top-traders' large trades only |
| 2 | Consensus Burst | `weather_whale_monitor.py` | 3+ traders buying the same side within 60 min |
| 3 | Rank Velocity | `leaderboard_analytics.py --velocity` | Who climbed fastest between two snapshots? |
| 5 | Accuracy Scorer | `weather_accuracy.py` | Historical win-rate on resolved weather markets |

Cells below run each module inline for quick exploration.

In [11]:
# ── DIAGNOSTIC: ping all APIs + 1-week trade scan ──────────────────────────
import time, requests
from datetime import datetime, timezone

GAMMA   = "https://gamma-api.polymarket.com"
DATA    = "https://data-api.polymarket.com"
CLOB    = "https://clob.polymarket.com"
CUTOFF  = int(time.time()) - 7 * 24 * 3600   # unix ts for "1 week ago"

def ping(label, url, params=None):
    try:
        t0 = time.time()
        r = requests.get(url, params=params or {}, timeout=10)
        ms = int((time.time() - t0) * 1000)
        status = "✅" if r.status_code == 200 else f"❌ {r.status_code}"
        print(f"  {status}  {label:<35}  {ms:>4}ms")
        return r.status_code == 200, r
    except Exception as e:
        print(f"  ❌  {label:<35}  ERROR: {e}")
        return False, None

print("=" * 60)
print("API CONNECTIVITY CHECK")
print("=" * 60)
ok1, r1 = ping("Gamma /events (weather)",     f"{GAMMA}/events",          {"tag_slug": "weather", "limit": 3})
ok2, r2 = ping("Data  /v1/leaderboard",        f"{DATA}/v1/leaderboard",   {"category": "WEATHER", "limit": 3})
ok3, r3 = ping("CLOB  /markets",               f"{CLOB}/markets",          {"limit": 1})

# ── 1. Active weather market count ──────────────────────────────────────────
print("\n" + "=" * 60)
print("WEATHER MARKETS (first page)")
print("=" * 60)
from weather_markets import WeatherMarketFetcher
fetcher = WeatherMarketFetcher(page_size=20, max_pages=1)
active = fetcher.fetch_active()
print(f"  Active weather markets found: {len(active)}")
for m in active[:5]:
    print(f"    • {m['question'][:70]}  vol=${m['volume']:,.0f}")

# ── 2. WEATHER leaderboard spot-check ───────────────────────────────────────
print("\n" + "=" * 60)
print("WEATHER LEADERBOARD (top 5)")
print("=" * 60)
leaders_raw = r2.json() if ok2 else []
leaders_raw = leaders_raw if isinstance(leaders_raw, list) else []
for r in leaders_raw[:5]:
    w = (r.get("proxyWallet") or "")[:14]
    print(f"  #{r.get('rank','?'):<3} {(r.get('userName') or ''):<20}  pnl=${r.get('pnl',0):>10,.2f}  {w}...")

# ── 3. Trades from last 1 week for top-5 wallets ────────────────────────────
print("\n" + "=" * 60)
print(f"TRADES IN LAST 1 WEEK  (since: {datetime.fromtimestamp(CUTOFF, tz=timezone.utc).strftime('%Y-%m-%d %H:%M')} UTC)")
print("=" * 60)
weather_cids = {m["conditionId"] for m in active if m.get("conditionId")}
top_wallets  = [r.get("proxyWallet") for r in leaders_raw[:5] if r.get("proxyWallet")]
recent_trades = []

for wallet in top_wallets:
    resp = requests.get(f"{DATA}/trades", params={"user": wallet, "takerOnly": "true", "limit": 100}, timeout=15)
    if resp.status_code != 200:
        continue
    for t in (resp.json() if isinstance(resp.json(), list) else []):
        ts = int(t.get("timestamp") or 0)
        if ts < CUTOFF:
            continue
        cid = t.get("market") or t.get("conditionId") or ""
        if cid not in weather_cids and weather_cids:
            continue
        notional = float(t.get("size") or 0) * float(t.get("price") or 0)
        recent_trades.append({
            "wallet": wallet[:14],
            "title":  (t.get("title") or "")[:55],
            "outcome": t.get("outcome"),
            "side":    t.get("side"),
            "price":   t.get("price"),
            "notional": round(notional, 2),
            "ts":      datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%m-%d %H:%M"),
        })
    time.sleep(0.1)

if recent_trades:
    recent_trades.sort(key=lambda x: x["notional"], reverse=True)
    print(f"  Found {len(recent_trades)} weather trades in last 1 week\n")
    print(f"  {'Date/Time':>11}  {'Notional':>10}  {'Side':>4}  {'Out':>4}  {'Price':>6}  Title")
    print("  " + "-" * 90)
    for t in recent_trades[:20]:
        print(f"  {t['ts']:>11}  ${t['notional']:>9,.2f}  {str(t['side']):>4}  {str(t['outcome']):>4}  {float(t['price'] or 0):>6.3f}  {t['title']}")
else:
    print("  No weather trades found in the last 1 week for top-5 traders.")
    print("  (This is normal — weather trades are infrequent; try a wider wallet set or window.)")

print("\n✅ Diagnostic complete")


API CONNECTIVITY CHECK
  ✅  Gamma /events (weather)                78ms
  ✅  Data  /v1/leaderboard                 160ms
  ✅  CLOB  /markets                        127ms

WEATHER MARKETS (first page)
  Active weather markets found: 58
    • Will there be at least 10000 measles cases in the U.S. in 2026?  vol=$6,473,572
    • Will 2026 be the fifth-hottest year on record?  vol=$664,240
    • 10.0 or above earthquake before 2027?  vol=$532,469
    • Will there be 8 or more earthquakes of magnitude 7.0 or higher worldwi  vol=$528,353
    • Will there be exactly 5 earthquakes of magnitude 7.0 or higher worldwi  vol=$446,573

WEATHER LEADERBOARD (top 5)
  #1   Unpluggedstoic        pnl=$  2,273.57  0xef988ffb5f3f...
  #2   gghff                 pnl=$    729.69  0x044f334595a7...
  #3   0x4addc66860b349959cb5070a83dbd9eeb6836fe5  pnl=$    690.06  0x4addc66860b3...

TRADES IN LAST 5 MINUTES  (cutoff: 09:30:39 UTC)
  No weather trades found in the last 5 minutes for top-5 traders.
  (This is n

In [12]:
# ── Cell 1: Active Weather Markets ──────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath("."))

from weather_markets import WeatherMarketFetcher

fetcher = WeatherMarketFetcher(page_size=50, max_pages=5)
print("Fetching active weather markets...")
active_markets = fetcher.fetch_active()
print(f"\nFound {len(active_markets)} active weather markets\n")
print(f"{'Question':<62} {'Volume':>12}  {'Bid':>6} {'Ask':>6}  Tags")
print("-" * 100)
for m in active_markets[:20]:
    tags = ",".join(m["tags"][:3])
    q = m["question"][:60]
    print(f"{q:<62} {m['volume']:>12,.0f}  {m['bestBid']:>6.3f} {m['bestAsk']:>6.3f}  {tags}")


Fetching active weather markets...

Found 2477 active weather markets

Question                                                             Volume     Bid    Ask  Tags
----------------------------------------------------------------------------------------------------
Will there be at least 10000 measles cases in the U.S. in 20      6,473,572   0.132  0.139  pandemics,science,measles
Will 2026 be the fifth-hottest year on record?                      664,240   0.010  0.014  science,pop-culture,global-temp
10.0 or above earthquake before 2027?                               532,469   0.049  0.055  science,weather,climate-science
Will there be 8 or more earthquakes of magnitude 7.0 or high        528,353   0.850  0.860  weather,pop-culture,science
Will there be exactly 5 earthquakes of magnitude 7.0 or high        446,573   0.000  0.001  weather,pop-culture,science
Will 2026 rank as the sixth-hottest year on record or lower?        434,323   0.024  0.027  science,pop-culture,global-temp
W

In [13]:
# ── Cell 2: Top WEATHER Leaderboard ─────────────────────────────────────────
import requests

resp = requests.get(
    "https://data-api.polymarket.com/v1/leaderboard",
    params={"category": "WEATHER", "timePeriod": "ALL", "orderBy": "PNL", "limit": 20},
    timeout=20,
)
leaders = resp.json() if resp.status_code == 200 else []
print(f"Top {len(leaders)} WEATHER traders (all-time by PnL)\n")
print(f"{'Rank':<6} {'Username':<22} {'PnL':>14}  {'Volume':>14}  Wallet")
print("-" * 80)
for r in leaders:
    wallet = (r.get("proxyWallet") or "")[:14] + "..."
    print(
        f"{r.get('rank', '?'):<6} {(r.get('userName') or ''):<22} "
        f"${r.get('pnl', 0):>13,.2f}  ${r.get('vol', 0):>13,.2f}  {wallet}"
    )


Top 20 WEATHER traders (all-time by PnL)

Rank   Username                          PnL          Volume  Wallet
--------------------------------------------------------------------------------
1      gopfan2                $   336,824.86  $ 4,545,440.77  0xf2f6af4f27ec...
2      aenews2                $   277,041.35  $ 9,970,055.89  0x44c1dfe43260...
3      gopfan                 $   118,426.47  $   739,900.98  0x6af75d4e4aaf...
4      ColdMath               $    95,432.04  $ 6,845,819.35  0x594edb9112f5...
5      bama124                $    86,600.55  $   410,556.07  0xe5c802623991...
6      Hans323                $    80,856.42  $ 6,934,963.79  0x0f37cb80dee4...
7      Handsanitizer23        $    71,379.08  $   888,456.38  0x05e70727a2e2...
8      automatedAItradingbot  $    63,986.22  $ 2,247,619.61  0xd8f8c13644ea...
9      BigMike11              $    62,776.72  $   850,755.65  0xecdbd79566a2...
10     Kapii                  $    57,190.13  $ 1,833,451.17  0xb74711992caf...
11     W

In [15]:
# ── Cell 3: Weather Whale Monitor — 1 loop, 1-week result filter ─────────────
# Idea 1 + 2: whale trades + consensus burst detection
# Runs 1 loop immediately; filters printed results to trades in last 1 week.
import time
from datetime import datetime, timezone
from weather_whale_monitor import (
    fetch_weather_leaderboard,
    poll_weather_wallets,
    detect_consensus_burst,
)
from weather_markets import WeatherMarketFetcher

ONE_WEEK_AGO = int(time.time()) - 7 * 24 * 3600

print("Fetching WEATHER leaderboard (top 15)...")
leaderboard = fetch_weather_leaderboard(limit=15)
wallets = [r["proxyWallet"] for r in leaderboard if r.get("proxyWallet")]
print(f"  Watching {len(wallets)} wallets\n")

print("Fetching active weather condition IDs...")
weather_cids = WeatherMarketFetcher(page_size=50, max_pages=3).get_active_condition_ids()
print(f"  Active weather markets: {len(weather_cids)}\n")

print("Polling trades (min notional $500)...")
alerts = poll_weather_wallets(wallets, weather_cids, min_notional=500)
print(f"  Total weather whale alerts found: {len(alerts)}")

# ── Filter to last 1 week ─────────────────────────────────────────────────────
recent = [a for a in alerts if int(a.get("timestamp") or 0) >= ONE_WEEK_AGO]
print(f"  Trades in last 1 week: {len(recent)}\n")

if recent:
    recent.sort(key=lambda x: x["notional"], reverse=True)
    print(f"{'Date/Time':>14}  {'Notional':>10}  {'Side':>4}  {'Outcome':>6}  {'Price':>6}  Title")
    print("-" * 96)
    for a in recent:
        ts_str = datetime.fromtimestamp(int(a["timestamp"]), tz=timezone.utc).strftime("%m-%d %H:%M")
        print(f"{ts_str:>14}  ${a['notional']:>9,.2f}  {str(a.get('side','')):>4}  "
              f"{str(a.get('outcome','')):>6}  {float(a.get('price') or 0):>6.3f}  "
              f"{(a.get('title') or '')[:55]}")
else:
    print("No whale-size weather trades in the last 1 week.")
    print("Showing most recent 10 alerts regardless of time:")
    for a in alerts[:10]:
        ts = int(a.get("timestamp") or 0)
        ts_str = datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M") if ts else "unknown"
        print(f"  [{ts_str}]  ${a['notional']:>9,.2f}  {str(a.get('outcome','')):>5}  {(a.get('title') or '')[:55]}")

# ── Consensus burst check ─────────────────────────────────────────────────────
print("\n── Consensus Burst Detection (min 2 traders, 1-week window) ──")
bursts = detect_consensus_burst(alerts, min_traders=2, window_minutes=7 * 24 * 60)
if bursts:
    for b in bursts[:5]:
        print(f"  🔥 BURST: {b['trader_count']} traders → {b['outcome']} on {(b.get('title') or '')[:50]}")
        print(f"           total notional=${b['total_notional']:,.2f}")
else:
    print("  No consensus bursts detected in current alert batch.")


Fetching WEATHER leaderboard (top 15)...
  Watching 15 wallets

Fetching active weather condition IDs...
  Active weather markets: 1415

Polling trades (min notional $500)...
  Total weather whale alerts found: 7
  Trades in last 1 week: 4

     Date/Time    Notional  Side  Outcome   Price  Title
------------------------------------------------------------------------------------------------
   04-02 11:20  $ 1,237.91   BUY      No   0.910  Will 2026 be the third-hottest year on record?
   04-04 09:32  $   788.99   BUY      No   0.981  Will the highest temperature in Moscow be 12°C on April
   04-02 11:20  $   550.46  SELL     Yes   0.283  Will 2026 be the hottest year on record?
   04-04 08:02  $   505.08   BUY      No   0.990  Will the highest temperature in Beijing be 13°C on Apri

── Consensus Burst Detection (min 2 traders, 1-week window) ──
  No consensus bursts detected in current alert batch.


## 🔬 Backtest: Whale + Consensus Buy Signals (3 months)

**Strategy being tested:** *"Follow top WEATHER traders' large buys and multi-trader convergences."*

For each signal detected in the lookback window the backtest asks:
> If you had copied that trade at the signal price, and the market later resolved — did you win?

| Signal type | Trigger | Entry |
|-------------|---------|-------|
| **WHALE** | Single top trader buys ≥ $100 notional | Entry price of that trade |
| **CONSENSUS** | ≥ 2 top traders buy same outcome within 24h | Average entry price of all trades in the burst |

**P&L model:**
- Win → `(1.0 − entry_price) × size`  *(you hold shares that resolve to $1)*
- Loss → `−entry_price × size`  *(shares expire at $0)*
- Return % → `+((1−p)/p × 100)` if win, `−100%` if loss

Only markets that have **fully resolved** contribute to the evaluation.
Open / unresolved markets are counted but excluded from win-rate/P&L stats.

In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# BACKTEST: Whale + Consensus buy signals on Weather markets (2-part)
#   Part A – Wallet-centric  : follow top-20 WEATHER leaderboard traders
#   Part B – Market-centric  : start from resolved markets → fetch all bets
# ══════════════════════════════════════════════════════════════════════════════
import sys, time, requests, json, random
from collections import defaultdict
from datetime import datetime, timezone, timedelta

sys.path.insert(0, ".")
from weather_markets import WeatherMarketFetcher, _parse_json_field, _determine_winner
from weather_whale_monitor import fetch_weather_leaderboard

DATA_BASE  = "https://data-api.polymarket.com"
GAMMA_BASE = "https://gamma-api.polymarket.com"

# ── Parameters ─────────────────────────────────────────────────────────────
MONTHS_BACK        = 3    # lookback window
MIN_NOTIONAL       = 10   # $ min per trade to count as a signal
WHALE_NOTIONAL     = 100  # $ threshold for "whale" label
CONSENSUS_TRADERS  = 2    # ≥ N wallets agreeing → consensus signal
CONSENSUS_WINDOW_H = 24   # hours window for consensus grouping
TOP_N_WALLETS      = 20   # WEATHER leaderboard wallets to track
MKTCENTRIC_SAMPLE  = 50   # resolved markets to sample for Part B

BT_CUTOFF = int((datetime.now(timezone.utc) - timedelta(days=MONTHS_BACK * 30)).timestamp())
print("=" * 65)
print("WEATHER MARKET BACKTEST")
print("=" * 65)
print(f"  Window    : {datetime.fromtimestamp(BT_CUTOFF, tz=timezone.utc).strftime('%Y-%m-%d')} → now  ({MONTHS_BACK} months)")
print(f"  Min $     : ${MIN_NOTIONAL}   Whale $: ${WHALE_NOTIONAL}   Wallets: {TOP_N_WALLETS}")

# ══════════════════════════════════════════════════════════════════════════════
# PART A – WALLET-CENTRIC
# ══════════════════════════════════════════════════════════════════════════════
print("\n━━━ PART A: Wallet-centric (follow top WEATHER PnL traders) ━━━━━━━━━")

# [A1] Build the 3-category market universe ───────────────────────────────
print("[A1] Building weather market universe …")
fetcher = WeatherMarketFetcher(page_size=50, max_pages=25)

# Category 1 – Resolved: closed events with a clear winner
closed_mkts  = fetcher.fetch_closed(max_pages=50)
resolved_map = {
    m["conditionId"]: m["resolvedOutcome"]
    for m in closed_mkts
    if m.get("conditionId") and m.get("resolvedOutcome")
}

# Category 2 – Active: market.active=True AND market.closed=False
active_mkts = fetcher.fetch_active()
active_cids = {m["conditionId"] for m in active_mkts if m.get("conditionId")}

# Category 3 – Pending: CLOB orderbook closed but parent event still running
#   (e.g. "Will 2026 be the hottest year?" — trading ended, resolves in 2027)
pending_cids = set()
raw_events = fetcher._fetch_events_paginated(extra_params={"active": "true"}, max_pages=25)
for ev in raw_events:
    for m in (ev.get("markets") or []):
        if m.get("closed") and not ev.get("closed") and m.get("conditionId"):
            pending_cids.add(m["conditionId"])

all_wx_cids = active_cids | set(resolved_map.keys()) | pending_cids

print(f"  Resolved (winner known)  : {len(resolved_map):,}")
print(f"  Active  (still trading)  : {len(active_cids):,}")
print(f"  Pending (CLOB closed,    : {len(pending_cids):,}  ← annual 2026 markets, resolving ~2027)")
print(f"  Total universe           : {len(all_wx_cids):,}")

# [A2] Collect wallet trade history ────────────────────────────────────────
print(f"\n[A2] Fetching {MONTHS_BACK}-month trade history for top-{TOP_N_WALLETS} WEATHER wallets …")
top_traders = fetch_weather_leaderboard(limit=TOP_N_WALLETS)
wallets     = [t["proxyWallet"] for t in top_traders if t.get("proxyWallet")]
all_hist    = []

for idx, wallet in enumerate(wallets, 1):
    wallet_trades = []
    for page in range(10):                         # up to 1 000 trades/wallet
        resp = requests.get(
            f"{DATA_BASE}/trades",
            params={"user": wallet, "takerOnly": "true",
                    "limit": 100, "offset": page * 100},
            timeout=20,
        )
        if resp.status_code != 200:
            break
        batch = resp.json() if isinstance(resp.json(), list) else []
        if not batch:
            break
        for t in batch:
            # NOTE: trade.market is always "" – use trade.conditionId
            cid = t.get("conditionId") or t.get("market") or ""
            ts  = int(t.get("timestamp") or 0)
            if cid in all_wx_cids and ts >= BT_CUTOFF:
                t["_wallet"] = wallet
                wallet_trades.append(t)
        oldest_ts = min((int(t.get("timestamp") or 0) for t in batch), default=0)
        if oldest_ts < BT_CUTOFF:
            break
        time.sleep(0.07)
    all_hist.extend(wallet_trades)
    if idx % 5 == 0:
        name = top_traders[idx-1].get("userName") or wallet[:12]
        print(f"  [{idx}/{len(wallets)}] {name:<20}  weather trades so far: {len(all_hist)}")
    time.sleep(0.10)

print(f"  Total: {len(all_hist)} weather trades from {len(wallets)} wallets")

# [A3] Build whale + consensus signals ────────────────────────────────────
print(f"\n[A3] Building signals (min ${MIN_NOTIONAL}) …")

whale_sigs = []
alert_feed = []  # all ≥ MIN_NOTIONAL BUY trades, used for consensus detection

for t in all_hist:
    try:
        price    = float(t.get("price") or 0)
        size     = float(t.get("size")  or 0)
        notional = price * size
    except (TypeError, ValueError):
        continue
    if notional < MIN_NOTIONAL or (t.get("side") or "").upper() != "BUY":
        continue

    cid     = t.get("conditionId") or t.get("market") or ""
    outcome = (t.get("outcome") or "").strip()
    category = (
        "resolved" if cid in resolved_map else
        "pending"  if cid in pending_cids  else
        "active"
    )
    sig = {
        "type": "WHALE", "conditionId": cid, "outcome": outcome,
        "price": price, "size": size, "notional": round(notional, 2),
        "timestamp": int(t.get("timestamp") or 0),
        "title":  (t.get("title") or "")[:65],
        "wallet": t.get("_wallet", ""),
        "category": category,
    }
    alert_feed.append(sig)
    if notional >= WHALE_NOTIONAL:
        whale_sigs.append(sig)

# Consensus: ≥ CONSENSUS_TRADERS wallets BUY same (cid, outcome) within window
groups = defaultdict(list)
for s in alert_feed:
    groups[(s["conditionId"], s["outcome"])].append(s)

consensus_sigs = []
for (cid, outcome), sigs in groups.items():
    sigs_s = sorted(sigs, key=lambda x: x["timestamp"])
    for i, anchor in enumerate(sigs_s):
        window_end  = anchor["timestamp"] + CONSENSUS_WINDOW_H * 3600
        in_window   = [s for s in sigs_s[i:] if s["timestamp"] <= window_end]
        uniq_wallets = {s["wallet"] for s in in_window}
        if len(uniq_wallets) >= CONSENSUS_TRADERS:
            category = (
                "resolved" if cid in resolved_map else
                "pending"  if cid in pending_cids  else "active"
            )
            consensus_sigs.append({
                "type": "CONSENSUS", "conditionId": cid, "outcome": outcome,
                "traderCount": len(uniq_wallets),
                "price":    sum(s["price"]    for s in in_window) / len(in_window),
                "size":     sum(s["size"]     for s in in_window),
                "notional": round(sum(s["notional"] for s in in_window), 2),
                "timestamp": anchor["timestamp"],
                "title":    anchor["title"],
                "wallet":   f"{len(uniq_wallets)} wallets",
                "category": category,
            })
            break

all_wallet_sigs = whale_sigs + consensus_sigs
print(f"  Whale signals  (≥${WHALE_NOTIONAL})       : {len(whale_sigs)}")
print(f"  Consensus sigs (≥{CONSENSUS_TRADERS} wallets, {CONSENSUS_WINDOW_H}h): {len(consensus_sigs)}")
print(f"  Total                           : {len(all_wallet_sigs)}")
for cat in ("resolved", "pending", "active"):
    nw = sum(1 for s in whale_sigs    if s.get("category") == cat)
    nc = sum(1 for s in consensus_sigs if s.get("category") == cat)
    print(f"    [{cat:8s}]  whale={nw}  consensus={nc}")

# [A4] Live Gamma resolution lookup for signals NOT already in resolved_map ──
unique_unresolved_cids = {
    s["conditionId"] for s in all_wallet_sigs
    if s["conditionId"] not in resolved_map
}
print(f"\n[A4] Live Gamma lookup for {len(unique_unresolved_cids)} unresolved conditionIds …")
live_resolved = {}
for i, cid in enumerate(unique_unresolved_cids):
    r = requests.get(f"{GAMMA_BASE}/markets", params={"conditionId": cid}, timeout=15)
    if r.status_code == 200:
        for m in (r.json() if isinstance(r.json(), list) else []):
            outcomes   = _parse_json_field(m.get("outcomes"), [])
            prices_raw = _parse_json_field(m.get("outcomePrices"), [])
            prices     = [float(p) for p in prices_raw] if prices_raw else []
            winner     = _determine_winner(outcomes, prices)
            if winner:
                live_resolved[cid] = winner
                break
    time.sleep(0.07)
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(unique_unresolved_cids)} checked …")

full_resolved = {**resolved_map, **live_resolved}
print(f"  Live lookup added  : {len(live_resolved)} new resolutions")
print(f"  Full resolved map  : {len(full_resolved):,}")

# [A5] Evaluate ────────────────────────────────────────────────────────────
print("\n[A5] Evaluating Part-A signals …")

def _eval(signals, resolved_lookup):
    ev, unresolved = [], []
    for sig in signals:
        winner = resolved_lookup.get(sig["conditionId"])
        if not winner:
            unresolved.append(sig)
            continue
        won    = (sig.get("outcome") or "").lower() == winner.lower()
        entry  = float(sig.get("price") or 0)
        size   = float(sig.get("size")  or 0)
        pnl    = (1.0 - entry) * size if won else -entry * size
        ret_pct = ((1.0 - entry) / entry * 100) if (won and entry > 0) else -100.0
        ev.append({**sig, "winner": winner, "won": won,
                   "pnl": round(pnl, 2), "pct_return": round(ret_pct, 1)})
    return ev, unresolved

ev_a, unresolved_a = _eval(all_wallet_sigs, full_resolved)
n_wins_a = sum(1 for e in ev_a if e["won"])

print(f"\n  ┌─────────────────────────────────────────────────────────────┐")
print(f"  │  PART A RESULTS  (wallet-centric, top-{TOP_N_WALLETS} WEATHER traders)  │")
print(f"  ├─────────────────────────────────────────────────────────────┤")
print(f"  │  Signals total  : {len(all_wallet_sigs):<42d}│")
print(f"  │  Evaluated      : {len(ev_a):<42d}│")
if ev_a:
    total_pnl_a = sum(e["pnl"]        for e in ev_a)
    avg_ret_a   = sum(e["pct_return"] for e in ev_a) / len(ev_a)
    avg_entry_a = sum(e["price"]      for e in ev_a) / len(ev_a)
    print(f"  │  Win rate       : {n_wins_a/len(ev_a)*100:.1f}%  ({n_wins_a}W / {len(ev_a)-n_wins_a}L)   [random=50%]          │")
    print(f"  │  Total P&L      : ${total_pnl_a:>+,.2f}{'':<35}│")
    print(f"  │  Avg return     : {avg_ret_a:>+.1f}%   Avg entry: {avg_entry_a:.3f}               │")
print(f"  │  Unresolved     : {len(unresolved_a):<42d}│")
if unresolved_a:
    by_cat = defaultdict(int)
    for s in unresolved_a:
        by_cat[s.get("category", "?")] += 1
    detail = "  ".join(f"{k}={v}" for k, v in sorted(by_cat.items()))
    print(f"  │    ({detail}){' ' * max(0, 57 - len(detail))}│")
print(f"  └─────────────────────────────────────────────────────────────┘")

if not ev_a and unresolved_a:
    titles = list({s["title"][:58] for s in unresolved_a if s.get("title")})[:4]
    print(f"\n  ⚠  All signals on unresolved long-duration markets.  Sample:")
    for t in titles:
        print(f"     • {t}")
    print(f"\n  Annual 2026 climate/earthquake markets resolve ~2027.")
    print(f"  → Part B (market-centric) produces evaluable results now.")

# ══════════════════════════════════════════════════════════════════════════════
# PART B – MARKET-CENTRIC  (guaranteed evaluable results)
# Start from resolved_map → fetch every trade on those markets → evaluate
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n━━━ PART B: Market-centric ({MKTCENTRIC_SAMPLE} resolved markets) ━━━━━━━━━━━━━━━")
print(f"[B1] Sampling {MKTCENTRIC_SAMPLE} resolved weather markets, fetching all trades ≥${MIN_NOTIONAL} …")

sample_pairs = list(resolved_map.items())
random.shuffle(sample_pairs)
sample_pairs = sample_pairs[:MKTCENTRIC_SAMPLE]

mkt_trades = []
for i, (cid, resolved_outcome) in enumerate(sample_pairs):
    for page in range(3):
        resp = requests.get(
            f"{DATA_BASE}/trades",
            params={"market": cid, "takerOnly": "true",
                    "limit": 100, "offset": page * 100},
            timeout=20,
        )
        if resp.status_code != 200:
            break
        batch = resp.json() if isinstance(resp.json(), list) else []
        if not batch:
            break
        for t in batch:
            notional = float(t.get("price") or 0) * float(t.get("size") or 0)
            ts       = int(t.get("timestamp") or 0)
            if notional >= MIN_NOTIONAL and (t.get("side") or "").upper() == "BUY":
                mkt_trades.append({
                    "conditionId":     cid,
                    "resolvedOutcome": resolved_outcome,
                    "wallet":          t.get("proxyWallet") or t.get("user") or "",
                    "outcome":         (t.get("outcome") or "").strip(),
                    "price":           float(t.get("price") or 0),
                    "size":            float(t.get("size")  or 0),
                    "notional":        round(notional, 2),
                    "timestamp":       ts,
                    "title":           (t.get("title") or "")[:65],
                })
        time.sleep(0.05)
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{MKTCENTRIC_SAMPLE}] {len(mkt_trades)} trades collected …")

print(f"  Total market trades collected: {len(mkt_trades)}")

# [B2] Evaluate ─────────────────────────────────────────────────────────────
ev_b = []
for t in mkt_trades:
    won    = (t.get("outcome") or "").lower() == (t.get("resolvedOutcome") or "").lower()
    entry  = t["price"]
    size   = t["size"]
    pnl    = (1.0 - entry) * size if won else -entry * size
    ret_pct = ((1.0 - entry) / entry * 100) if (won and entry > 0) else -100.0
    ev_b.append({**t, "won": won, "pnl": round(pnl, 2), "pct_return": round(ret_pct, 1)})

n_wins_b    = sum(1 for e in ev_b if e["won"])
total_pnl_b = sum(e["pnl"]        for e in ev_b)
avg_ret_b   = (sum(e["pct_return"] for e in ev_b) / len(ev_b)) if ev_b else 0
avg_entry_b = (sum(e["price"]      for e in ev_b) / len(ev_b)) if ev_b else 0

yes_bets = [e for e in ev_b if (e.get("outcome") or "").lower() == "yes"]
no_bets  = [e for e in ev_b if (e.get("outcome") or "").lower() == "no"]
yes_wins = sum(1 for e in yes_bets if e["won"])
no_wins  = sum(1 for e in no_bets  if e["won"])

print(f"\n  ┌─────────────────────────────────────────────────────────────────┐")
print(f"  │  PART B RESULTS  ({len(ev_b)} trades on {MKTCENTRIC_SAMPLE} resolved weather markets)  │")
print(f"  ├─────────────────────────────────────────────────────────────────┤")
print(f"  │  Evaluated   : {len(ev_b):<50d}│")
wr_str = f"{n_wins_b/len(ev_b)*100:.1f}%  ({n_wins_b}W/{len(ev_b)-n_wins_b}L)  [random baseline=50%]" if ev_b else "N/A"
print(f"  │  Win rate    : {wr_str:<50}│")
print(f"  │  Total P&L   : ${total_pnl_b:>+,.2f}{'':<46}│")
print(f"  │  Avg return  : {avg_ret_b:>+.1f}%   Avg entry price: {avg_entry_b:.3f}              │")
print(f"  ├─────────────────────────────────────────────────────────────────┤")
if yes_bets:
    yw = f"{yes_wins/len(yes_bets)*100:.1f}% ({yes_wins}W/{len(yes_bets)-yes_wins}L)"
    print(f"  │  YES bets: {len(yes_bets):<4d}  win={yw:<40}│")
if no_bets:
    nw_str = f"{no_wins/len(no_bets)*100:.1f}% ({no_wins}W/{len(no_bets)-no_wins}L)"
    print(f"  │  NO  bets: {len(no_bets):<4d}  win={nw_str:<40}│")
print(f"  └─────────────────────────────────────────────────────────────────┘")

# [B3] Detail table: top-30 largest bets ──────────────────────────────────
print(f"\n  Top 30 largest bets on resolved weather markets:")
print(f"  {'Date':>11}  {'Bet':>4}  {'Result':>6}  {'Entry':>6}  {'$Notional':>10}  {'P&L':>9}  {'Title'}")
print("  " + "─" * 88)
for e in sorted(ev_b, key=lambda x: x["notional"], reverse=True)[:30]:
    ts_s   = datetime.fromtimestamp(e["timestamp"], tz=timezone.utc).strftime("%m-%d %H:%M") if e["timestamp"] else "???"
    symbol = "✓" if e["won"] else "✗"
    print(
        f"  {ts_s:>11}  {e['outcome']:>4}  "
        f"{symbol}{e['resolvedOutcome']:>5}  "
        f"{e['price']:>6.3f}  ${e['notional']:>9,.2f}  "
        f"${e['pnl']:>+8,.2f}  {e['title'][:38]}"
    )
if len(ev_b) > 30:
    print(f"  … {len(ev_b)-30} more rows")
print("  " + "─" * 88)

# [B4] Insight summary ──────────────────────────────────────────────────────
print(f"\n  INSIGHTS:")
if ev_b:
    # Do big bets (whale-size ≥$100) perform better than small bets?
    big_bets   = [e for e in ev_b if e["notional"] >= WHALE_NOTIONAL]
    small_bets = [e for e in ev_b if e["notional"] <  WHALE_NOTIONAL]
    if big_bets:
        bw = sum(1 for e in big_bets if e["won"])
        print(f"  • Whale bets (≥${WHALE_NOTIONAL}):   {len(big_bets)} trades  "
              f"win={bw/len(big_bets)*100:.1f}%  "
              f"pnl=${sum(e['pnl'] for e in big_bets):+,.2f}")
    if small_bets:
        sw = sum(1 for e in small_bets if e["won"])
        print(f"  • Small bets (<${WHALE_NOTIONAL}):  {len(small_bets)} trades  "
              f"win={sw/len(small_bets)*100:.1f}%  "
              f"pnl=${sum(e['pnl'] for e in small_bets):+,.2f}")

    # Most profitable market
    mkt_pnl = defaultdict(float)
    mkt_title = {}
    for e in ev_b:
        mkt_pnl[e["conditionId"]] += e["pnl"]
        mkt_title[e["conditionId"]] = e.get("title", "")[:55]
    best_cid  = max(mkt_pnl, key=mkt_pnl.get)
    worst_cid = min(mkt_pnl, key=mkt_pnl.get)
    print(f"  • Most profitable market : ${mkt_pnl[best_cid]:>+,.2f}  {mkt_title[best_cid]}")
    print(f"  • Least profitable market: ${mkt_pnl[worst_cid]:>+,.2f}  {mkt_title[worst_cid]}")
else:
    print("  No trades found for the sampled resolved markets in this window.")
    print("  Try increasing MKTCENTRIC_SAMPLE or decreasing MONTHS_BACK.")


WEATHER MARKET BACKTEST
  Window    : 2026-01-04 → now  (3 months)
  Min $     : $10   Whale $: $100   Wallets: 20

━━━ PART A: Wallet-centric (follow top WEATHER PnL traders) ━━━━━━━━━
[A1] Building weather market universe …
  Resolved (winner known)  : 20,109
  Active  (still trading)  : 2,642
  Pending (CLOB closed,    : 0  ← annual 2026 markets, resolving ~2027)
  Total universe           : 22,751

[A2] Fetching 3-month trade history for top-20 WEATHER wallets …
  [5/20] bama124               weather trades so far: 937
  [10/20] Kapii                 weather trades so far: 1262
  [15/20] junkbonds             weather trades so far: 2398
  [20/20] aboss                 weather trades so far: 4694
  Total: 4694 weather trades from 20 wallets

[A3] Building signals (min $10) …
  Whale signals  (≥$100)       : 1353
  Consensus sigs (≥2 wallets, 24h): 53
  Total                           : 1406
    [resolved]  whale=916  consensus=51
    [pending ]  whale=0  consensus=0
    [active  ]  

In [18]:
# ── Cell 4: Trader Accuracy Scorer ──────────────────────────────────────────
# Idea 5: score top WEATHER traders by win-rate on resolved markets
# Note: first run may be slow (~1–2 min) due to paginating closed markets
#       and fetching trade history per wallet.
from weather_accuracy import WeatherAccuracyAnalyzer

analyzer = WeatherAccuracyAnalyzer(top_n=20)   # score top-20 for demo speed
results = analyzer.run()
analyzer.print_table(results, top_n=20)

# Optional: save to DB + CSV
# analyzer.save_to_db(results, db_path="logs/weather_accuracy.db")
# analyzer.save_to_csv(results, csv_path="logs/weather_accuracy.csv")


[accuracy] Fetching closed weather markets...
[accuracy] Closed markets fetched: 6433 | with resolvable outcome: 6431
[accuracy] Fetching top WEATHER leaderboard traders...
[accuracy] Scoring 20 traders...
  [ 1/20] ⚠️ gopfan2              trades=   0  win_rate=  0.0%  weighted=  0.0%
  [ 2/20] ⚠️ aenews2              trades=   0  win_rate=  0.0%  weighted=  0.0%
  [ 3/20] ⚠️ gopfan               trades=   0  win_rate=  0.0%  weighted=  0.0%
  [ 4/20] ⚠️ ColdMath             trades=   0  win_rate=  0.0%  weighted=  0.0%
  [ 5/20] ⚠️ bama124              trades=   0  win_rate=  0.0%  weighted=  0.0%
  [ 6/20] ⚠️ Hans323              trades=   0  win_rate=  0.0%  weighted=  0.0%
  [ 7/20] ⚠️ Handsanitizer23      trades=   0  win_rate=  0.0%  weighted=  0.0%
  [ 8/20] ⚠️ automatedAItradingbot trades=   0  win_rate=  0.0%  weighted=  0.0%
  [ 9/20] ⚠️ BigMike11            trades=   0  win_rate=  0.0%  weighted=  0.0%
  [10/20]   Kapii                trades= 195  win_rate=100.0%  weighted=1

### 🗒️ Rank Velocity (Idea 3) — CLI-only (requires daemon snapshots)

The rank velocity feature compares two leaderboard snapshots captured over time.
It's intentionally a **CLI tool** because it needs the daemon to have run at least twice
(minimum 2 snapshots in `logs/leaderboard.db`).

**Step 1** — Start the snapshot daemon (runs in background, seeds every 6 hours):
```bash
python3 weather_snapshot_daemon.py &
```

**Step 2** — After ≥2 snapshots exist, run the velocity alert:
```bash
python3 leaderboard_analytics.py \
  --db logs/leaderboard.db \
  --category WEATHER \
  --time-period ALL \
  --velocity \
  --min-jump 5
```

Output shows traders who climbed ≥5 rank positions, plus their current live positions and recent trades fetched from the Data API.

In [20]:
# ── Extended summary with deeper insights ────────────────────────────────────
print("=" * 68)
print("BACKTEST SUMMARY")
print("=" * 68)

# ── Part A ──────────────────────────────────────────────────────────────────
print(f"\nPART A  (wallet-centric — top-{TOP_N_WALLETS} WEATHER PnL traders)")
print(f"  signals   : {len(all_wallet_sigs):,}   evaluated: {len(ev_a):,}   unresolved: {len(unresolved_a):,}")
if ev_a:
    n_wins_a   = sum(1 for e in ev_a if e["won"])
    pnl_a      = sum(e["pnl"]        for e in ev_a)
    avg_entry_a = sum(e["price"]     for e in ev_a) / len(ev_a)
    print(f"  win rate  : {n_wins_a}/{len(ev_a)} = {n_wins_a/len(ev_a)*100:.1f}%")
    print(f"  total P&L : ${pnl_a:>+,.2f}")
    print(f"  avg entry : {avg_entry_a:.3f}  ← {avg_entry_a*100:.0f}% implied probability")
    # high entry means near-certain bets; true edge = win_rate - entry_price
    edge = n_wins_a/len(ev_a) - avg_entry_a
    print(f"  edge      : {edge:>+.3f}  (win_rate − avg_entry; >0 = alpha)")
    for sig_type in ("WHALE", "CONSENSUS"):
        sub = [e for e in ev_a if e.get("type") == sig_type]
        if sub:
            sw = sum(1 for e in sub if e["won"])
            ae = sum(e["price"] for e in sub) / len(sub)
            sp = sum(e["pnl"] for e in sub)
            print(f"  {sig_type:9s} : {len(sub):>4d} sigs  wr={sw/len(sub)*100:.1f}%  "
                  f"avg_entry={ae:.3f}  pnl=${sp:>+,.2f}")

# ── Part A: filter to "uncertain" bets only (entry < 0.7) ──────────────────
uncertain_a = [e for e in ev_a if e["price"] < 0.70]
if uncertain_a:
    uw = sum(1 for e in uncertain_a if e["won"])
    ae = sum(e["price"] for e in uncertain_a) / len(uncertain_a)
    edge_u = uw/len(uncertain_a) - ae
    print(f"\n  ── 'Uncertain' bets only (entry < 0.70):")
    print(f"     count     : {len(uncertain_a)}")
    print(f"     win rate  : {uw/len(uncertain_a)*100:.1f}%")
    print(f"     avg entry : {ae:.3f}")
    print(f"     edge      : {edge_u:>+.3f}")
    print(f"     pnl       : ${sum(e['pnl'] for e in uncertain_a):>+,.2f}")
else:
    print("\n  No bets with entry < 0.70 (all near-certain bets).")

# ── Part B ──────────────────────────────────────────────────────────────────
print(f"\nPART B  (market-centric — {MKTCENTRIC_SAMPLE} random resolved markets)")
print(f"  trades    : {len(mkt_trades):,}   evaluated: {len(ev_b):,}")
if ev_b:
    n_wins_b   = sum(1 for e in ev_b if e["won"])
    pnl_b      = sum(e["pnl"]    for e in ev_b)
    avg_entry_b = sum(e["price"] for e in ev_b) / len(ev_b)
    edge_b     = n_wins_b/len(ev_b) - avg_entry_b
    print(f"  win rate  : {n_wins_b}/{len(ev_b)} = {n_wins_b/len(ev_b)*100:.1f}%")
    print(f"  total P&L : ${pnl_b:>+,.2f}")
    print(f"  avg entry : {avg_entry_b:.3f}  ← {avg_entry_b*100:.0f}% implied prob")
    print(f"  edge      : {edge_b:>+.3f}")

    # Filter uncertain
    uncertain_b = [e for e in ev_b if e["price"] < 0.70]
    if uncertain_b:
        uw = sum(1 for e in uncertain_b if e["won"])
        ae = sum(e["price"] for e in uncertain_b) / len(uncertain_b)
        eu = uw/len(uncertain_b) - ae
        up = sum(e["pnl"] for e in uncertain_b)
        print(f"\n  ── 'Uncertain' bets only (entry < 0.70):")
        print(f"     count    : {len(uncertain_b)}")
        print(f"     win rate : {uw/len(uncertain_b)*100:.1f}%")
        print(f"     avg entry: {ae:.3f}")
        print(f"     edge     : {eu:>+.3f}")
        print(f"     pnl      : ${up:>+,.2f}")

    # By bet side
    yes_b = [e for e in ev_b if (e.get("outcome") or "").lower() == "yes"]
    no_b  = [e for e in ev_b if (e.get("outcome") or "").lower() == "no"]
    yw = sum(1 for e in yes_b if e["won"]) if yes_b else 0
    nw = sum(1 for e in no_b  if e["won"]) if no_b  else 0
    print(f"\n  Bet-side breakdown:")
    if yes_b:
        ae_y = sum(e["price"] for e in yes_b)/len(yes_b)
        print(f"    YES: {len(yes_b):>4d} trades  wr={yw/len(yes_b)*100:.1f}%  "
              f"avg_entry={ae_y:.3f}  edge={yw/len(yes_b)-ae_y:>+.3f}  "
              f"pnl=${sum(e['pnl'] for e in yes_b):>+,.2f}")
    if no_b:
        ae_n = sum(e["price"] for e in no_b)/len(no_b)
        print(f"    NO:  {len(no_b):>4d} trades  wr={nw/len(no_b)*100:.1f}%  "
              f"avg_entry={ae_n:.3f}  edge={nw/len(no_b)-ae_n:>+.3f}  "
              f"pnl=${sum(e['pnl'] for e in no_b):>+,.2f}")

# ── Universe stats ──────────────────────────────────────────────────────────
print(f"\nUniverse (built during run):")
print(f"  resolved_map  : {len(resolved_map):,} markets  (closed events with winner)")
print(f"  active_cids   : {len(active_cids):,} markets  (still trading)")
print(f"  pending_cids  : {len(pending_cids):,}  (CLOB closed, event still open)")
print(f"  all_wx_cids   : {len(all_wx_cids):,}")
print("=" * 68)


BACKTEST SUMMARY

PART A  (wallet-centric — top-20 WEATHER PnL traders)
  signals   : 1,406   evaluated: 1,406   unresolved: 0
  win rate  : 1222/1406 = 86.9%
  total P&L : $+98,482.27
  avg entry : 0.846  ← 85% implied probability
  edge      : +0.023  (win_rate − avg_entry; >0 = alpha)
  WHALE     : 1353 sigs  wr=88.2%  avg_entry=0.861  pnl=$+85,885.28
  CONSENSUS :   53 sigs  wr=54.7%  avg_entry=0.470  pnl=$+12,596.99

  ── 'Uncertain' bets only (entry < 0.70):
     count     : 282
     win rate  : 43.3%
     avg entry : 0.390
     edge      : +0.043
     pnl       : $+71,546.98

PART B  (market-centric — 50 random resolved markets)
  trades    : 2,571   evaluated: 2,571
  win rate  : 2356/2571 = 91.6%
  total P&L : $-2,226.69
  avg entry : 0.910  ← 91% implied prob
  edge      : +0.007

  ── 'Uncertain' bets only (entry < 0.70):
     count    : 274
     win rate : 33.2%
     avg entry: 0.389
     edge     : -0.057
     pnl      : $-2,894.47

  Bet-side breakdown:
    YES:  611 trad

## 📊 Deep Analysis — Logs, Backtest Diagnostics & Next Steps

The cells below:
1. Parse `logs/price_data.csv` and `logs/alerts.csv` to surface what the bot has captured so far.
2. Diagnose the mid-price vs ask-price bug in the arbitrage scanner.
3. Analyse the backtest numbers from Part A & B with fee-adjusted edge.
4. Run a longer-period backtest with uncertainty filter and fee model.


In [21]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL A — logs/ Inspection
# Parses price_data.csv and alerts.csv to understand what the bot has captured.
# ═══════════════════════════════════════════════════════════════════════════
import csv, os
from datetime import datetime, timezone
from collections import defaultdict

PRICE_CSV  = "logs/price_data.csv"
ALERTS_CSV = "logs/alerts.csv"

# ── 1. Price data summary ────────────────────────────────────────────────────
print("=" * 70)
print("PRICE DATA LOG ANALYSIS  (logs/price_data.csv)")
print("=" * 70)

price_rows = []
if os.path.exists(PRICE_CSV):
    with open(PRICE_CSV, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        price_rows = list(reader)

print(f"  Total rows      : {len(price_rows):,}")
if price_rows:
    timestamps = [r["timestamp"] for r in price_rows if r.get("timestamp")]
    print(f"  Date range      : {min(timestamps)[:19]}  →  {max(timestamps)[:19]}")
    
    arb_rows = [r for r in price_rows if r.get("arbitrage_opportunity") == "1"]
    print(f"  Arb opportunities: {len(arb_rows):,}  ({len(arb_rows)/len(price_rows)*100:.1f}%)")
    
    total_costs = [float(r["total_cost"]) for r in price_rows if r.get("total_cost")]
    print(f"  Avg total_cost  : {sum(total_costs)/len(total_costs):.6f}  (should be <1.0 to flag arb)")
    print(f"  Min total_cost  : {min(total_costs):.6f}")
    print()
    print("  ⚠️  All total_costs = 1.0000 → BUG CONFIRMED: bot is summing mid-prices,")
    print("     not ask-prices. Mid-prices always sum to ~1.0 by market design.")
    print("     Fix: use yes_ask + no_ask for arbitrage check (not yes_price + no_price).")
    
    # Count unique markets
    unique_mkts = {r["market_id"] for r in price_rows}
    print(f"\n  Unique markets scanned: {len(unique_mkts):,}")
    
    # Category distribution (guess from question text)
    categories = defaultdict(int)
    for r in price_rows:
        q = (r.get("market_question") or "").lower()
        if any(k in q for k in ["nhl","nba","mlb","nfl","soccer","champions","stanley","basketball","football"]):
            categories["sports"] += 1
        elif any(k in q for k in ["btc","bitcoin","eth","crypto","coin"]):
            categories["crypto"] += 1
        elif any(k in q for k in ["trump","president","congress","senate","election","vote"]):
            categories["politics"] += 1
        elif any(k in q for k in ["hurricane","temperature","weather","climate","storm","earthquake"]):
            categories["weather"] += 1
        else:
            categories["other"] += 1
    print(f"\n  Category breakdown (keyword-inferred):")
    for cat, cnt in sorted(categories.items(), key=lambda x: -x[1]):
        print(f"    {cat:<12}: {cnt:>5} rows  ({cnt/len(price_rows)*100:.1f}%)")


PRICE DATA LOG ANALYSIS  (logs/price_data.csv)
  Total rows      : 649
  Date range      : 2026-03-22T17:22:44  →  2026-03-22T17:39:16
  Arb opportunities: 0  (0.0%)
  Avg total_cost  : 1.000000  (should be <1.0 to flag arb)
  Min total_cost  : 1.000000

  ⚠️  All total_costs = 1.0000 → BUG CONFIRMED: bot is summing mid-prices,
     not ask-prices. Mid-prices always sum to ~1.0 by market design.
     Fix: use yes_ask + no_ask for arbitrage check (not yes_price + no_price).

  Unique markets scanned: 100

  Category breakdown (keyword-inferred):
    sports      :   329 rows  (50.7%)
    other       :   288 rows  (44.4%)
    crypto      :    26 rows  (4.0%)
    politics    :     6 rows  (0.9%)


In [22]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL B — Alerts Log Analysis
# ═══════════════════════════════════════════════════════════════════════════
import csv, os
from collections import defaultdict

ALERTS_CSV = "logs/alerts.csv"

print("=" * 70)
print("ALERTS LOG ANALYSIS  (logs/alerts.csv)")
print("=" * 70)

alert_rows = []
if os.path.exists(ALERTS_CSV):
    with open(ALERTS_CSV, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        alert_rows = list(reader)

print(f"  Total alerts    : {len(alert_rows):,}")
if alert_rows:
    notionals = []
    for r in alert_rows:
        try:
            notionals.append(float(r.get("notional") or 0))
        except (ValueError, TypeError):
            pass

    if notionals:
        print(f"  Total notional  : ${sum(notionals):>14,.2f}")
        print(f"  Avg notional    : ${sum(notionals)/len(notionals):>14,.2f}")
        print(f"  Max notional    : ${max(notionals):>14,.2f}")
        print(f"  Min notional    : ${min(notionals):>14,.2f}")

    # Top wallets by total notional
    wallet_totals = defaultdict(float)
    wallet_counts = defaultdict(int)
    for r in alert_rows:
        w = (r.get("proxyWallet") or "")[:16]
        try:
            wallet_totals[w] += float(r.get("notional") or 0)
            wallet_counts[w] += 1
        except (ValueError, TypeError):
            pass
    print(f"\n  Top 5 wallets by total notional:")
    for w, total in sorted(wallet_totals.items(), key=lambda x: -x[1])[:5]:
        print(f"    {w}...  {wallet_counts[w]:>4} trades  ${total:>12,.0f}")

    # Outcome distribution
    outcome_dist = defaultdict(lambda: {"count": 0, "notional": 0.0})
    for r in alert_rows:
        outcome = (r.get("outcome") or "Unknown").strip()
        try:
            n = float(r.get("notional") or 0)
        except (ValueError, TypeError):
            n = 0.0
        outcome_dist[outcome]["count"] += 1
        outcome_dist[outcome]["notional"] += n
    print(f"\n  Outcome breakdown (top 10 by count):")
    for outcome, stats in sorted(outcome_dist.items(), key=lambda x: -x[1]["count"])[:10]:
        print(f"    {outcome[:30]:<30}  {stats['count']:>4} trades  ${stats['notional']:>12,.0f}")

    # Side distribution
    sides = defaultdict(int)
    for r in alert_rows:
        sides[(r.get("side") or "?").upper()] += 1
    print(f"\n  Trade side:  {dict(sides)}")

    # Weather vs other
    weather_kw = ["hurricane","storm","temperature","celsius","fahrenheit",
                  "rainfall","tornado","earthquake","snowfall","weather",
                  "climate","hottest","coldest","warmest","gistemp","tropical"]
    weather_alerts = [r for r in alert_rows if any(k in (r.get("title") or "").lower() for k in weather_kw)]
    print(f"\n  Weather alerts in general log: {len(weather_alerts):,}")
    print(f"  Sports/other alerts          : {len(alert_rows)-len(weather_alerts):,}")
    print()
    print("  ℹ️  General whale monitor runs on ALL categories (sports dominant).")
    print("     Weather-specific trades go to logs/alerts.db (weather_whale_monitor.py).")


ALERTS LOG ANALYSIS  (logs/alerts.csv)
  Total alerts    : 262
  Total notional  : $ 14,360,907.41
  Avg notional    : $     54,812.62
  Max notional    : $    447,453.67
  Min notional    : $      5,031.89

  Top 5 wallets by total notional:
    0x03e8a544e97eef...   101 trades  $   5,761,960
    0x2a2c53bd278c04...    68 trades  $   4,008,806
    0x492442eab586f2...    78 trades  $   3,926,570
    0x6ac5bb06a9eb05...    14 trades  $     649,813
    0xead152b855effa...     1 trades  $      13,758

  Outcome breakdown (top 10 by count):
    Over                              51 trades  $   2,874,065
    Under                             24 trades  $   2,003,537
    76ers                             19 trades  $   1,044,216
    Duke Blue Devils                  13 trades  $     289,564
    Thunder                           12 trades  $     630,632
    Warriors                           9 trades  $     793,681
    Blue Jackets                       9 trades  $     284,375
    Rockets     

In [23]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL C — Fee-Adjusted Backtest Analysis
# Re-analyses the backtest results already in memory (ev_a, ev_b) by
# subtracting Polymarket's 2% taker fee from every trade.
# ═══════════════════════════════════════════════════════════════════════════

TAKER_FEE = 0.02   # 2% on the amount wagered (per leg)

def fee_adjusted_pnl(entry: float, size: float, won: bool) -> float:
    """
    Polymarket charges 2% of the trade value (price * size) as taker fee.
    fee = entry * size * TAKER_FEE
    net_pnl = gross_pnl - fee
    """
    fee = entry * size * TAKER_FEE
    gross = (1.0 - entry) * size if won else (-entry * size)
    return gross - fee

def fee_adjusted_edge(trades, resolved_lookup=None):
    """Re-evaluate a list of trade dicts with 2% fee model."""
    results = []
    for t in trades:
        # Support both ev_a (has 'won' already) and raw trade lists
        if resolved_lookup is not None:
            winner = resolved_lookup.get(t.get("conditionId", ""))
            if not winner:
                continue
            won = (t.get("outcome") or "").lower() == winner.lower()
        else:
            won = t.get("won", False)

        entry = float(t.get("price", 0))
        size  = float(t.get("size", 0))
        net   = fee_adjusted_pnl(entry, size, won)
        results.append({**t, "won": won, "fee_pnl": round(net, 2)})
    return results

print("=" * 70)
print("FEE-ADJUSTED BACKTEST ANALYSIS  (2% taker fee per trade)")
print("=" * 70)

# ── Part A ─────────────────────────────────────────────────────────────────
if 'ev_a' in globals() and ev_a:
    adj_a = fee_adjusted_edge(ev_a)
    wins_a = sum(1 for e in adj_a if e["won"])
    pnl_adj_a = sum(e["fee_pnl"] for e in adj_a)
    gross_a   = sum(e["pnl"]     for e in adj_a)
    fee_total_a = gross_a - pnl_adj_a

    print(f"\nPART A — Wallet-centric (top-{TOP_N_WALLETS} WEATHER traders)")
    print(f"  Gross P&L (no fees) : ${gross_a:>+12,.2f}")
    print(f"  Fees paid           : ${fee_total_a:>+12,.2f}")
    print(f"  Net P&L (after fee) : ${pnl_adj_a:>+12,.2f}  ←  {'✅ profitable' if pnl_adj_a > 0 else '❌ unprofitable after fees'}")
    
    # Edge check per signal type
    for sig_type in ("WHALE", "CONSENSUS"):
        sub = [e for e in adj_a if e.get("type") == sig_type]
        if sub:
            sw = sum(1 for e in sub if e["won"])
            ae = sum(e["price"] for e in sub) / len(sub)
            sp = sum(e["fee_pnl"] for e in sub)
            edge_fee = sw/len(sub) - ae - ae*TAKER_FEE
            print(f"  {sig_type:9s}: {len(sub):>4} sigs  wr={sw/len(sub)*100:.1f}%  "
                  f"avg_entry={ae:.3f}  fee_edge={edge_fee:>+.3f}  net_pnl=${sp:>+,.2f}")

    # Uncertain bets after fee
    unc_a = [e for e in adj_a if float(e.get("price", 1)) < 0.70]
    if unc_a:
        uw = sum(1 for e in unc_a if e["won"])
        ae = sum(e["price"] for e in unc_a) / len(unc_a)
        up = sum(e["fee_pnl"] for e in unc_a)
        fee_edge_u = uw/len(unc_a) - ae - ae*TAKER_FEE
        print(f"\n  'Uncertain' bets (entry < 0.70) after fees:")
        print(f"     count     : {len(unc_a)}")
        print(f"     win rate  : {uw/len(unc_a)*100:.1f}%")
        print(f"     avg entry : {ae:.3f}")
        print(f"     fee_edge  : {fee_edge_u:>+.3f}  ← {'✅ positive' if fee_edge_u > 0 else '❌ negative'}")
        print(f"     net pnl   : ${up:>+,.2f}")
else:
    print("\nPart A results (ev_a) not in memory. Run the backtest cell first.")

# ── Part B ─────────────────────────────────────────────────────────────────
if 'ev_b' in globals() and ev_b:
    adj_b = fee_adjusted_edge(ev_b)
    wins_b = sum(1 for e in adj_b if e["won"])
    pnl_adj_b = sum(e["fee_pnl"] for e in adj_b)
    gross_b   = sum(e.get("pnl", 0) for e in ev_b)
    fee_total_b = gross_b - pnl_adj_b

    print(f"\nPART B — Market-centric ({MKTCENTRIC_SAMPLE} resolved markets)")
    print(f"  Gross P&L (no fees) : ${gross_b:>+12,.2f}")
    print(f"  Fees paid           : ${fee_total_b:>+12,.2f}")
    print(f"  Net P&L (after fee) : ${pnl_adj_b:>+12,.2f}  ←  {'✅ profitable' if pnl_adj_b > 0 else '❌ unprofitable after fees'}")
    
    # By bet side
    for side_label in ("yes", "no"):
        sub = [e for e in adj_b if (e.get("outcome") or "").lower() == side_label]
        if sub:
            sw = sum(1 for e in sub if e["won"])
            ae = sum(e["price"] for e in sub)/len(sub)
            sp = sum(e["fee_pnl"] for e in sub)
            fee_edge = sw/len(sub) - ae - ae*TAKER_FEE
            print(f"  {side_label.upper():<4} bets: {len(sub):>4}  wr={sw/len(sub)*100:.1f}%  "
                  f"avg_entry={ae:.3f}  fee_edge={fee_edge:>+.3f}  net_pnl=${sp:>+,.2f}")
else:
    print("\nPart B results (ev_b) not in memory. Run the backtest cell first.")

print(f"\n{'─'*70}")
print(f"CONCLUSION:")
print(f"  • Once 2% fees are included, the edge shrinks dramatically.")
print(f"  • Only CONSENSUS signals on 'uncertain' bets (entry 30-70%) show")
print(f"    meaningful positive edge after fees — worth isolating & backtesting further.")
print(f"  • The NO-bet bias in Part B is real but tiny (+1.1% gross) — fee eats it.")


FEE-ADJUSTED BACKTEST ANALYSIS  (2% taker fee per trade)

PART A — Wallet-centric (top-20 WEATHER traders)
  Gross P&L (no fees) : $  +98,482.27
  Fees paid           : $  +31,349.39
  Net P&L (after fee) : $  +67,132.88  ←  ✅ profitable
  WHALE    : 1353 sigs  wr=88.2%  avg_entry=0.861  fee_edge=+0.004  net_pnl=$+55,541.35
  CONSENSUS:   53 sigs  wr=54.7%  avg_entry=0.470  fee_edge=+0.068  net_pnl=$+11,591.53

  'Uncertain' bets (entry < 0.70) after fees:
     count     : 282
     win rate  : 43.3%
     avg entry : 0.390
     fee_edge  : +0.035  ← ✅ positive
     net pnl   : $+69,214.26

PART B — Market-centric (50 resolved markets)
  Gross P&L (no fees) : $   -2,226.69
  Fees paid           : $   +9,198.06
  Net P&L (after fee) : $  -11,424.75  ←  ❌ unprofitable after fees
  YES  bets:  611  wr=82.8%  avg_entry=0.833  fee_edge=-0.022  net_pnl=$-5,136.89
  NO   bets: 1960  wr=94.4%  avg_entry=0.933  fee_edge=-0.008  net_pnl=$-6,287.86

─────────────────────────────────────────────────

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL D — Longer-Period Backtest (12 months, uncertainty-filtered, fee-adjusted)
# Robust version: per-request retry + backoff, graceful skip on timeout,
# progress shown every 10 markets so you can see it working.
# ═══════════════════════════════════════════════════════════════════════════
import sys, time, requests, random
from collections import defaultdict
from datetime import datetime, timezone, timedelta

sys.path.insert(0, ".")
from weather_markets import WeatherMarketFetcher

DATA_BASE_LT  = "https://data-api.polymarket.com"
GAMMA_BASE_LT = "https://gamma-api.polymarket.com"

# ── Tunable parameters ────────────────────────────────────────────────────────
LT_MONTHS_BACK   = 12    # lookback window
LT_MIN_NOTIONAL  = 10    # $ min per trade to count
LT_SAMPLE        = 300   # resolved markets to sample  (300 → ~500 YES unc trades, tighter CI)
LT_TAKER_FEE     = 0.02  # 2% Polymarket taker fee
LT_ENTRY_LOW     = 0.30  # uncertainty band lower bound
LT_ENTRY_HIGH    = 0.70  # uncertainty band upper bound
REQUEST_TIMEOUT  = 30    # seconds per HTTP call  (was 20 — too tight)
MAX_RETRIES      = 3     # attempts before skipping a market
SLEEP_BETWEEN    = 0.10  # seconds between market requests (rate-limit)

LT_CUTOFF = int((datetime.now(timezone.utc) - timedelta(days=LT_MONTHS_BACK * 30)).timestamp())
print("=" * 72)
print(f"EXTENDED BACKTEST  ({LT_MONTHS_BACK}-month, uncertainty-filtered, fee-adjusted)")
print("=" * 72)
print(f"  Window   : {datetime.fromtimestamp(LT_CUTOFF, tz=timezone.utc).strftime('%Y-%m-%d')} → now")
print(f"  Markets  : {LT_SAMPLE} resolved weather markets (random sample)")
print(f"  Filter   : entry price {LT_ENTRY_LOW:.0%}–{LT_ENTRY_HIGH:.0%}  (uncertain band only)")
print(f"  Fee      : {LT_TAKER_FEE:.0%} taker fee deducted")
print(f"  Timeout  : {REQUEST_TIMEOUT}s per call,  {MAX_RETRIES} retries with backoff\n")


# ── Robust HTTP helper ────────────────────────────────────────────────────────
# NOTE: session is shared across retries to reuse TCP connections.
_lt_session = requests.Session()

def _safe_get(url: str, params: dict, timeout: int = REQUEST_TIMEOUT,
              max_retries: int = MAX_RETRIES) -> list:
    """
    GET with retry + exponential backoff.
    Returns parsed JSON list, or [] on unrecoverable error (timeout / 5xx / etc.).
    Never raises — caller just gets an empty list and continues.
    """
    for attempt in range(max_retries):
        try:
            resp = _lt_session.get(url, params=params, timeout=timeout)
            if resp.status_code == 429:                        # rate-limited
                wait = 2.0 * (2 ** attempt)
                print(f"    ⚠ 429 rate-limit — waiting {wait:.0f}s …", flush=True)
                time.sleep(wait)
                continue
            if resp.status_code in (500, 502, 503, 504):      # server errors
                if attempt < max_retries - 1:
                    time.sleep(0.8 * (2 ** attempt))
                    continue
                return []
            resp.raise_for_status()
            data = resp.json()
            return data if isinstance(data, list) else []
        except requests.exceptions.Timeout:
            if attempt < max_retries - 1:
                wait = 1.5 * (2 ** attempt)
                print(f"    ⚠ Timeout (attempt {attempt+1}/{max_retries}) — retrying in {wait:.0f}s …", flush=True)
                time.sleep(wait)
            else:
                print(f"    ✗ Giving up on this request after {max_retries} timeouts.", flush=True)
                return []
        except requests.exceptions.ConnectionError:
            if attempt < max_retries - 1:
                time.sleep(1.5 * (2 ** attempt))
            else:
                return []
        except Exception:
            return []
    return []


# ── Step 1: Build resolved market map ────────────────────────────────────────
print("[1] Fetching resolved weather markets from Gamma (max_pages=20) …")
fetcher_lt = WeatherMarketFetcher(page_size=50, max_pages=20, timeout=REQUEST_TIMEOUT)
closed_lt  = fetcher_lt.fetch_closed(max_pages=20)
resolved_lt = {
    m["conditionId"]: m["resolvedOutcome"]
    for m in closed_lt
    if m.get("conditionId") and m.get("resolvedOutcome")
}
print(f"  Resolved markets available: {len(resolved_lt):,}")
if len(resolved_lt) < LT_SAMPLE:
    print(f"  ⚠ Only {len(resolved_lt)} resolved markets found — "
          f"reducing sample to {len(resolved_lt)}.")
    LT_SAMPLE = len(resolved_lt)


# ── Step 2: Sample markets + fetch all BUY trades ────────────────────────────
print(f"\n[2] Sampling {LT_SAMPLE} markets and fetching trades …")
pairs = list(resolved_lt.items())
random.shuffle(pairs)
pairs = pairs[:LT_SAMPLE]

lt_trades   = []
skipped_mkt = 0

for i, (cid, resolved_outcome) in enumerate(pairs):
    mkt_ok = True
    for page in range(3):                           # max 300 trades/market
        batch = _safe_get(
            f"{DATA_BASE_LT}/trades",
            {"market": cid, "takerOnly": "true",
             "limit": 100, "offset": page * 100},
        )
        if not batch:                               # empty or error — stop paging
            if page == 0:
                skipped_mkt += 1
                mkt_ok = False
            break
        for t in batch:
            notional = float(t.get("price") or 0) * float(t.get("size") or 0)
            ts       = int(t.get("timestamp") or 0)
            if (notional >= LT_MIN_NOTIONAL
                    and (t.get("side") or "").upper() == "BUY"
                    and ts >= LT_CUTOFF):
                lt_trades.append({
                    "conditionId":     cid,
                    "resolvedOutcome": resolved_outcome,
                    "wallet":          t.get("proxyWallet") or "",
                    "outcome":         (t.get("outcome") or "").strip(),
                    "price":           float(t.get("price") or 0),
                    "size":            float(t.get("size")  or 0),
                    "notional":        round(notional, 2),
                    "timestamp":       ts,
                    "title":           (t.get("title") or "")[:65],
                })
        if len(batch) < 100:
            break                                   # last page — no need to paginate
        time.sleep(SLEEP_BETWEEN)

    if (i + 1) % 10 == 0 or (i + 1) == LT_SAMPLE:
        print(f"  [{i+1:>3}/{LT_SAMPLE}]  trades so far: {len(lt_trades):,}  "
              f"skipped markets: {skipped_mkt}", flush=True)
    time.sleep(SLEEP_BETWEEN)

print(f"\n  ✓ Done.  Trades collected: {len(lt_trades):,}  "
      f"(markets skipped due to errors: {skipped_mkt})")


# ── Step 3: Evaluate ─────────────────────────────────────────────────────────
lt_all_ev = []
for t in lt_trades:
    won   = t["outcome"].lower() == (t["resolvedOutcome"] or "").lower()
    entry = t["price"]
    size  = t["size"]
    gross = (1.0 - entry) * size if won else -entry * size
    fee   = entry * size * LT_TAKER_FEE
    lt_all_ev.append({**t, "won": won,
                      "gross_pnl": round(gross, 2),
                      "fee":       round(fee, 2),
                      "net_pnl":   round(gross - fee, 2)})

lt_unc = [e for e in lt_all_ev if LT_ENTRY_LOW <= e["price"] <= LT_ENTRY_HIGH]


# ── Step 4: Print results ─────────────────────────────────────────────────────
def _summarise(label, trades):
    if not trades:
        print(f"\n  {label}: no trades found.")
        return
    n    = len(trades)
    wins = sum(1 for t in trades if t["won"])
    wr   = wins / n
    avg_e = sum(t["price"]     for t in trades) / n
    gross = sum(t["gross_pnl"] for t in trades)
    fees  = sum(t["fee"]       for t in trades)
    net   = sum(t["net_pnl"]   for t in trades)
    edge     = wr - avg_e
    fee_edge = edge - avg_e * LT_TAKER_FEE
    w = max(0, 52 - len(label))
    print(f"\n  ┌─── {label} {'─'*w}┐")
    print(f"  │  Trades       : {n:>7,}                                 │")
    print(f"  │  Win rate     : {wr*100:>6.1f}%  ({wins}W / {n-wins}L)              │")
    print(f"  │  Avg entry    : {avg_e:>6.3f}  (implied prob {avg_e*100:.0f}%)         │")
    print(f"  │  Gross edge   : {edge:>+.3f}  (win_rate − avg_entry)         │")
    print(f"  │  Fee-adj edge : {fee_edge:>+.3f}  ({'✅ alpha' if fee_edge > 0 else '❌ no edge after fees'})               │")
    print(f"  │  Gross P&L    : ${gross:>+12,.2f}                          │")
    print(f"  │  Fees paid    : ${fees:>+12,.2f}                          │")
    print(f"  │  NET P&L      : ${net:>+12,.2f}  ({'✅' if net > 0 else '❌'})                  │")
    print(f"  └──────────────────────────────────────────────────────────┘")

print("\n" + "=" * 72)
print("RESULTS")
print("=" * 72)
_summarise(f"ALL TRADES  ({LT_MONTHS_BACK}-month, {LT_SAMPLE} markets)", lt_all_ev)
_summarise(f"UNCERTAIN BAND  (entry {LT_ENTRY_LOW:.0%}–{LT_ENTRY_HIGH:.0%})", lt_unc)

print(f"\n  Uncertainty-band breakdown by bet side:")
for side in ("yes", "no"):
    sub = [e for e in lt_unc if e["outcome"].lower() == side]
    if sub:
        wins = sum(1 for e in sub if e["won"])
        ae   = sum(e["price"] for e in sub) / len(sub)
        net  = sum(e["net_pnl"] for e in sub)
        fe   = wins/len(sub) - ae - ae * LT_TAKER_FEE
        print(f"    {side.upper():<3}: {len(sub):>4} trades  wr={wins/len(sub)*100:.1f}%  "
              f"avg_entry={ae:.3f}  fee_edge={fe:>+.3f}  net=${net:>+,.2f}")


# ── Step 5: Bootstrap 95% CI ──────────────────────────────────────────────────
# Uses 2 000 bootstrap resamples; percentile indices are 0-based.
# Lower bound = index 49  (50th value  / 2000 ≈ 2.50th percentile)
# Upper bound = index 1949 (1950th value / 2000 ≈ 97.50th percentile)  ← fixed (was 1950)
if lt_unc:
    import random as _rng
    wins_arr = [1 if e["won"] else 0 for e in lt_unc]
    boot_wr  = sorted(
        sum(_rng.choices(wins_arr, k=len(wins_arr))) / len(wins_arr)
        for _ in range(2000)
    )
    ci_lo = boot_wr[49]    # 2.5th  percentile  (index 49  of 2000)
    ci_hi = boot_wr[1949]  # 97.5th percentile  (index 1949 of 2000)  ← was 1950
    actual_wr     = sum(wins_arr) / len(wins_arr)
    avg_entry_unc = sum(e["price"] for e in lt_unc) / len(lt_unc)
    print(f"\n  Bootstrap 95% CI on win-rate (uncertain band, n={len(lt_unc)}):")
    print(f"    Observed   : {actual_wr*100:.2f}%")
    print(f"    95% CI     : [{ci_lo*100:.2f}%, {ci_hi*100:.2f}%]")
    print(f"    Implied p  : {avg_entry_unc*100:.2f}%")
    if ci_lo > avg_entry_unc:
        print(f"    ✅ Edge is statistically significant")
    elif ci_hi > avg_entry_unc:
        print(f"    ⚠️  Edge is borderline — CI spans the implied probability")
    else:
        print(f"    ❌ No statistically significant edge in uncertain band")

print(f"\n{'─'*72}")
print("TIP: Re-run Cells E, G, H, I after this completes to refresh all downstream analysis.")


EXTENDED BACKTEST  (12-month, uncertainty-filtered, fee-adjusted)
  Window   : 2025-04-12 → now
  Markets  : 300 resolved weather markets (random sample)
  Filter   : entry price 30%–70%  (uncertain band only)
  Fee      : 2% taker fee deducted
  Timeout  : 30s per call,  3 retries with backoff

[1] Fetching resolved weather markets from Gamma (max_pages=20) …
  Resolved markets available: 6,431

[2] Sampling 300 markets and fetching trades …
  [ 10/300]  trades so far: 297  skipped markets: 0
  [ 20/300]  trades so far: 667  skipped markets: 0
  [ 30/300]  trades so far: 1,295  skipped markets: 0
  [ 40/300]  trades so far: 1,507  skipped markets: 0
  [ 50/300]  trades so far: 1,895  skipped markets: 0
  [ 60/300]  trades so far: 2,254  skipped markets: 0
  [ 70/300]  trades so far: 2,409  skipped markets: 0
  [ 80/300]  trades so far: 2,997  skipped markets: 0
  [ 90/300]  trades so far: 3,261  skipped markets: 0
  [100/300]  trades so far: 3,604  skipped markets: 0
  [110/300]  trad

In [31]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL E — Walk-Forward Validation (train 9 months → validate 3 months)
# Checks whether the uncertainty-band edge is stable across time
# (requires ev_b or lt_all_ev from CELL D above)
# ═══════════════════════════════════════════════════════════════════════════
from datetime import datetime, timezone, timedelta
from collections import defaultdict

TAKER_FEE_WF = 0.02
ENTRY_LOW_WF  = 0.30
ENTRY_HIGH_WF = 0.70

def _wf_summarise(label, trades, fee=TAKER_FEE_WF):
    if not trades:
        return {"label": label, "n": 0, "wr": 0, "avg_entry": 0, "net_pnl": 0, "fee_edge": 0}
    n    = len(trades)
    wins = sum(1 for t in trades if t["won"])
    wr   = wins / n
    ae   = sum(t["price"] for t in trades) / n
    net  = sum(t["net_pnl"] if "net_pnl" in t else (
               (1-t["price"])*t["size"] - t["price"]*t["size"]*fee if t["won"]
               else -t["price"]*t["size"] - t["price"]*t["size"]*fee)
           for t in trades)
    fe   = wr - ae - ae * fee
    return {"label": label, "n": n, "wr": wr, "avg_entry": ae, "net_pnl": net, "fee_edge": fe}

# Use lt_all_ev from CELL D, or fall back to ev_b from original backtest
source = None
if 'lt_all_ev' in globals() and lt_all_ev:
    source = lt_all_ev
    source_label = f"{LT_MONTHS_BACK}-month extended backtest"
elif 'ev_b' in globals() and ev_b:
    # ev_b doesn't have net_pnl; add it
    TAKER_FEE_WF_LOCAL = 0.02
    source = []
    for e in ev_b:
        entry = e["price"]; size = e["size"]
        gross = (1-entry)*size if e["won"] else -entry*size
        fee   = entry * size * TAKER_FEE_WF_LOCAL
        source.append({**e, "net_pnl": gross - fee})
    source_label = "3-month original backtest"
else:
    print("No backtest data in memory. Run CELL D (or the original backtest) first.")
    source = []
    source_label = "none"

if source:
    now_ts  = int(datetime.now(timezone.utc).timestamp())
    months  = LT_MONTHS_BACK if 'LT_MONTHS_BACK' in globals() else MONTHS_BACK
    start_ts = now_ts - months * 30 * 24 * 3600
    span     = now_ts - start_ts

    # Split into 3 roughly equal folds
    fold_size = span // 3
    folds = []
    for k in range(3):
        lo = start_ts + k * fold_size
        hi = lo + fold_size
        fold_trades = [t for t in source
                       if lo <= int(t.get("timestamp", 0)) < hi]
        folds.append(fold_trades)

    print("=" * 72)
    print(f"WALK-FORWARD VALIDATION  (source: {source_label})")
    print("=" * 72)
    print(f"  Split into 3 time-folds, each ~{fold_size//86400} days\n")

    print(f"  {'Fold':<14} {'All trades':>11}  {'Unc trades':>11}  {'WR (unc)':>10}  {'Fee edge':>10}  {'Net PnL':>12}")
    print(f"  {'─'*14} {'─'*11}  {'─'*11}  {'─'*10}  {'─'*10}  {'─'*12}")

    for k, fold in enumerate(folds):
        fold_unc = [t for t in fold if ENTRY_LOW_WF <= float(t.get("price",0)) <= ENTRY_HIGH_WF]
        all_s = _wf_summarise(f"Fold {k+1}", fold)
        unc_s = _wf_summarise(f"Fold {k+1} unc", fold_unc)
        wr_str = f"{unc_s['wr']*100:.1f}%" if unc_s['n'] > 0 else "N/A"
        fe_str = f"{unc_s['fee_edge']:>+.3f}" if unc_s['n'] > 0 else "N/A"
        pnl_str = f"${unc_s['net_pnl']:>+,.0f}" if unc_s['n'] > 0 else "N/A"
        print(f"  Fold {k+1:<9} {all_s['n']:>11,}  {unc_s['n']:>11,}  {wr_str:>10}  {fe_str:>10}  {pnl_str:>12}")

    # Overall uncertainty-band summary
    all_unc = [t for t in source if ENTRY_LOW_WF <= float(t.get("price",0)) <= ENTRY_HIGH_WF]
    total_s = _wf_summarise("TOTAL", all_unc)
    print(f"  {'─'*14} {'─'*11}  {'─'*11}  {'─'*10}  {'─'*10}  {'─'*12}")
    print(f"  {'TOTAL':<14} {len(source):>11,}  {len(all_unc):>11,}  "
          f"{total_s['wr']*100:.1f}% {' ':>4}  {total_s['fee_edge']:>+.3f} {' ':>4}  ${total_s['net_pnl']:>+,.0f}")

    print(f"\n  Interpretation:")
    pos_folds = sum(1 for k, fold in enumerate(folds)
                    if [t for t in fold if ENTRY_LOW_WF <= float(t.get("price",0)) <= ENTRY_HIGH_WF]
                    and _wf_summarise("x", [t for t in fold if ENTRY_LOW_WF <= float(t.get("price",0)) <= ENTRY_HIGH_WF])["fee_edge"] > 0)
    print(f"    Folds with positive fee-edge: {pos_folds}/3")
    if pos_folds >= 2:
        print(f"    ✅ Edge appears consistent across time — strategy worth paper-trading")
    elif pos_folds == 1:
        print(f"    ⚠️  Edge is intermittent — more data needed before going live")
    else:
        print(f"    ❌ Edge does not persist across folds — do NOT trade this signal live")


WALK-FORWARD VALIDATION  (source: 12-month extended backtest)
  Split into 3 time-folds, each ~120 days

  Fold            All trades   Unc trades    WR (unc)    Fee edge       Net PnL
  ────────────── ───────────  ───────────  ──────────  ──────────  ────────────
  Fold 1               3,606          436       63.5%      +0.084       $+8,387
  Fold 2               3,790          466       55.4%      -0.002       $+8,956
  Fold 3               1,808          218       45.9%      -0.086       $-4,310
  ────────────── ───────────  ───────────  ──────────  ──────────  ────────────
  TOTAL                9,205        1,120  56.7%       +0.015       $+13,033

  Interpretation:
    Folds with positive fee-edge: 1/3
    ⚠️  Edge is intermittent — more data needed before going live


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════
# CELL G — YES-Only Uncertain Band: Walk-Forward Breakdown
# Tests whether the +8.6% fee-edge on YES uncertain bets (30–70% entry)
# holds across all 3 time folds or is driven entirely by Fold 2.
# Also runs a bootstrap 95% CI per fold to assess statistical significance.
# ═══════════════════════════════════════════════════════════════════════════
import random as _rng
from datetime import datetime, timezone

# ── Config (mirrors Cell D / Cell E settings) ─────────────────────────────
ENTRY_LOW_G   = 0.30
ENTRY_HIGH_G  = 0.70
TAKER_FEE_G   = 0.02

# ── Source data check ──────────────────────────────────────────────────────
if "lt_all_ev" not in globals() or not lt_all_ev:
    print("❌  lt_all_ev not found — run Cell D first.")
else:
    # ── Rebuild time folds (same logic as Cell E) ──────────────────────────
    now_ts_g  = int(datetime.now(timezone.utc).timestamp())
    months_g  = LT_MONTHS_BACK if "LT_MONTHS_BACK" in globals() else 12
    start_ts_g = now_ts_g - months_g * 30 * 24 * 3600
    span_g     = now_ts_g - start_ts_g
    fold_size_g = span_g // 3

    def _fold_label(k):
        lo = datetime.fromtimestamp(start_ts_g + k * fold_size_g, tz=timezone.utc)
        hi = datetime.fromtimestamp(start_ts_g + (k+1) * fold_size_g, tz=timezone.utc)
        return f"{lo.strftime('%b %Y')} – {hi.strftime('%b %Y')}"

    folds_g = []
    for k in range(3):
        lo = start_ts_g + k * fold_size_g
        hi = lo + fold_size_g
        folds_g.append([t for t in lt_all_ev if lo <= t.get("timestamp", 0) < hi])

    # ── Helper ─────────────────────────────────────────────────────────────
    def _stats(trades, label=""):
        n = len(trades)
        if n == 0:
            return {"n": 0, "wr": 0, "avg_e": 0, "fee_edge": 0, "net_pnl": 0}
        wins  = sum(1 for t in trades if t["won"])
        wr    = wins / n
        avg_e = sum(t["price"] for t in trades) / n
        net   = sum(t["net_pnl"] for t in trades)
        fee_edge = wr - avg_e - avg_e * TAKER_FEE_G
        return {"n": n, "wr": wr, "avg_e": avg_e, "fee_edge": fee_edge, "net_pnl": net}

    def _boot_ci(trades, n_boot=3000):
        """
        Bootstrap 95% CI on win-rate.
        Lower bound = index int(0.025 * n_boot)       ≈ 2.53rd percentile
        Upper bound = index int(0.975 * n_boot) - 1   = exact 97.50th percentile
        Using n_boot-1 denominator keeps both bounds symmetric.
        """
        if len(trades) < 5:
            return None, None
        arr = [1 if t["won"] else 0 for t in trades]
        boot = sorted(
            sum(_rng.choices(arr, k=len(arr))) / len(arr) for _ in range(n_boot)
        )
        lo_idx = int(0.025 * n_boot)        # ≈ 2.5th percentile
        hi_idx = int(0.975 * n_boot) - 1   # exact 97.5th percentile (0-based)
        hi_idx = max(lo_idx + 1, hi_idx)    # safety: ensure hi > lo
        return boot[lo_idx], boot[hi_idx]

    # ── Segment the data ───────────────────────────────────────────────────
    # YES uncertain: entry 30-70%, outcome == "yes"
    # NO  uncertain: entry 30-70%, outcome == "no"
    yes_unc_all = [t for t in lt_all_ev
                   if ENTRY_LOW_G <= t["price"] <= ENTRY_HIGH_G
                   and t["outcome"].lower() == "yes"]
    no_unc_all  = [t for t in lt_all_ev
                   if ENTRY_LOW_G <= t["price"] <= ENTRY_HIGH_G
                   and t["outcome"].lower() == "no"]

    yes_folds = []
    no_folds  = []
    for k in range(3):
        lo = start_ts_g + k * fold_size_g
        hi = lo + fold_size_g
        yes_folds.append([t for t in yes_unc_all if lo <= t.get("timestamp", 0) < hi])
        no_folds.append( [t for t in no_unc_all  if lo <= t.get("timestamp", 0) < hi])

    # ── Print: YES walk-forward ────────────────────────────────────────────
    print("=" * 76)
    print("CELL G — YES-ONLY UNCERTAIN BAND WALK-FORWARD  (entry 30–70%, outcome=YES)")
    print("=" * 76)
    print(f"  Total YES uncertain trades (12-month): {len(yes_unc_all)}")
    print()
    print(f"  {'Fold':<22} {'n':>5}  {'WR':>7}  {'Implied':>8}  {'Raw edge':>9}  {'Fee edge':>9}  {'Net PnL':>10}  {'95% CI WR'}")
    print(f"  {'─'*22} {'─'*5}  {'─'*7}  {'─'*8}  {'─'*9}  {'─'*9}  {'─'*10}  {'─'*20}")

    yes_pos_folds = 0
    for k, fold in enumerate(yes_folds):
        s = _stats(fold)
        ci_lo, ci_hi = _boot_ci(fold)
        ci_str = f"[{ci_lo*100:.1f}%, {ci_hi*100:.1f}%]" if ci_lo is not None else "N/A"
        sig_flag = ""
        if ci_lo is not None and ci_lo > s["avg_e"]:
            sig_flag = " ✅"
        elif ci_lo is not None and ci_hi > s["avg_e"]:
            sig_flag = " ⚠️"
        else:
            sig_flag = " ❌"
        if s["fee_edge"] > 0:
            yes_pos_folds += 1
        n_str    = f"{s['n']:>5}"
        wr_str   = f"{s['wr']*100:>6.1f}%" if s["n"] > 0 else "   N/A"
        imp_str  = f"{s['avg_e']*100:>7.1f}%" if s["n"] > 0 else "    N/A"
        raw_str  = f"{(s['wr']-s['avg_e']):>+.3f}" if s["n"] > 0 else "    N/A"
        fe_str   = f"{s['fee_edge']:>+.3f}" if s["n"] > 0 else "    N/A"
        pnl_str  = f"${s['net_pnl']:>+,.0f}" if s["n"] > 0 else "    N/A"
        print(f"  Fold {k+1} {_fold_label(k):<16} {n_str}  {wr_str}  {imp_str}  {raw_str}  {fe_str}  {pnl_str:>10}  {ci_str}{sig_flag}")

    # Total YES row
    s_tot = _stats(yes_unc_all)
    ci_lo_t, ci_hi_t = _boot_ci(yes_unc_all)
    ci_str_t = f"[{ci_lo_t*100:.1f}%, {ci_hi_t*100:.1f}%]" if ci_lo_t else "N/A"
    sig_t = " ✅" if (ci_lo_t and ci_lo_t > s_tot["avg_e"]) else (" ⚠️" if (ci_lo_t and ci_hi_t > s_tot["avg_e"]) else " ❌")
    print(f"  {'─'*22} {'─'*5}  {'─'*7}  {'─'*8}  {'─'*9}  {'─'*9}  {'─'*10}  {'─'*20}")
    print(f"  {'TOTAL YES UNC':<22} {s_tot['n']:>5}  {s_tot['wr']*100:>6.1f}%  {s_tot['avg_e']*100:>7.1f}%  "
          f"{(s_tot['wr']-s_tot['avg_e']):>+.3f}  {s_tot['fee_edge']:>+.3f}  ${s_tot['net_pnl']:>+,.0f}  {ci_str_t}{sig_t}")

    # ── Print: NO walk-forward (for contrast) ─────────────────────────────
    print()
    print("─" * 76)
    print("  NO-ONLY UNCERTAIN BAND (entry 30–70%, outcome=NO)  — shown for contrast")
    print("─" * 76)
    print(f"  Total NO  uncertain trades (12-month): {len(no_unc_all)}")
    print()
    print(f"  {'Fold':<22} {'n':>5}  {'WR':>7}  {'Implied':>8}  {'Fee edge':>9}  {'Net PnL':>10}")
    print(f"  {'─'*22} {'─'*5}  {'─'*7}  {'─'*8}  {'─'*9}  {'─'*10}")
    for k, fold in enumerate(no_folds):
        s = _stats(fold)
        wr_s  = f"{s['wr']*100:>6.1f}%" if s["n"] > 0 else "   N/A"
        imp_s = f"{s['avg_e']*100:>7.1f}%" if s["n"] > 0 else "    N/A"
        fe_s  = f"{s['fee_edge']:>+.3f}" if s["n"] > 0 else "    N/A"
        pnl_s = f"${s['net_pnl']:>+,.0f}" if s["n"] > 0 else "    N/A"
        print(f"  Fold {k+1} {_fold_label(k):<16} {s['n']:>5}  {wr_s}  {imp_s}  {fe_s}  {pnl_s:>10}")
    s_no = _stats(no_unc_all)
    print(f"  {'─'*22} {'─'*5}  {'─'*7}  {'─'*8}  {'─'*9}  {'─'*10}")
    print(f"  {'TOTAL NO UNC':<22} {s_no['n']:>5}  {s_no['wr']*100:>6.1f}%  "
          f"{s_no['avg_e']*100:>7.1f}%  {s_no['fee_edge']:>+.3f}  ${s_no['net_pnl']:>+,.0f}")

    # ── YES/NO divergence by fold ──────────────────────────────────────────
    print()
    print("─" * 76)
    print("  YES vs NO FEE-EDGE DIVERGENCE PER FOLD")
    print("─" * 76)
    print(f"  {'Fold':<22}  {'YES fee_edge':>13}  {'NO fee_edge':>12}  {'Gap':>8}")
    print(f"  {'─'*22}  {'─'*13}  {'─'*12}  {'─'*8}")
    for k in range(3):
        sy = _stats(yes_folds[k])
        sn = _stats(no_folds[k])
        ye = sy["fee_edge"] if sy["n"] > 0 else float("nan")
        ne = sn["fee_edge"] if sn["n"] > 0 else float("nan")
        gap = ye - ne if sy["n"] > 0 and sn["n"] > 0 else float("nan")
        print(f"  Fold {k+1} {_fold_label(k):<16}  {ye:>+13.3f}  {ne:>+12.3f}  {gap:>+8.3f}")

    # ── Final verdict ──────────────────────────────────────────────────────
    print()
    print("=" * 76)
    print("VERDICT")
    print("=" * 76)
    print(f"  YES uncertain folds with positive fee-edge: {yes_pos_folds}/3")
    print()
    if yes_pos_folds == 3:
        print("  ✅ YES-only edge is CONSISTENT across all 3 folds.")
        print("     → Strong case for paper-trading YES uncertain signals immediately.")
    elif yes_pos_folds == 2:
        print("  ⚠️  YES-only edge holds in 2/3 folds — likely real but season-dependent.")
        print("     → Paper-trade with a seasonal filter (check which fold failed).")
    elif yes_pos_folds == 1:
        print("  ⚠️  YES-only edge holds in only 1/3 folds.")
        print("     → Fold 2 was likely an outlier. Do NOT trade live yet.")
        print("     → Next step: increase LT_SAMPLE to 300 for more statistical power.")
    else:
        print("  ❌ YES-only edge does not hold across any fold.")
        print("     → The +8.6% aggregate figure was driven by a single fold.")
        print("     → Do not trade this signal. Re-examine market selection criteria.")

    print()
    if ci_lo_t is not None:
        if ci_lo_t > s_tot["avg_e"]:
            print(f"  Bootstrap CI (all YES unc, n={s_tot['n']}): [{ci_lo_t*100:.1f}%, {ci_hi_t*100:.1f}%]")
            print(f"  ✅ Statistically significant: entire CI is above implied prob {s_tot['avg_e']*100:.1f}%")
        elif ci_hi_t > s_tot["avg_e"]:
            print(f"  Bootstrap CI (all YES unc, n={s_tot['n']}): [{ci_lo_t*100:.1f}%, {ci_hi_t*100:.1f}%]")
            print(f"  ⚠️  CI straddles implied prob {s_tot['avg_e']*100:.1f}% — borderline significant")
            print(f"     Increase sample size (LT_SAMPLE=300) to get tighter bounds.")
        else:
            print(f"  ❌ CI [{ci_lo_t*100:.1f}%, {ci_hi_t*100:.1f}%] is entirely below implied {s_tot['avg_e']*100:.1f}%")

    print()
    print("  RECOMMENDED NEXT ACTION:")
    print("  • If 2/3+ folds positive AND CI significant → paper-trade YES uncertain + CONSENSUS combo")
    print("  • If only 1/3 folds positive                → run LT_SAMPLE=300 re-sample first")
    print("  • In both cases: apply BUG-1 fix (yes_ask, not yes_price) before any live orders")


CELL G — YES-ONLY UNCERTAIN BAND WALK-FORWARD  (entry 30–70%, outcome=YES)
  Total YES uncertain trades (12-month): 559

  Fold                       n       WR   Implied   Raw edge   Fee edge     Net PnL  95% CI WR
  ────────────────────── ─────  ───────  ────────  ─────────  ─────────  ──────────  ────────────────────
  Fold 1 Apr 2025 – Aug 2025   223    60.1%     46.9%  +0.132  +0.123     $+8,915  [53.8%, 66.4%] ✅
  Fold 2 Aug 2025 – Dec 2025   231    50.6%     52.5%  -0.018  -0.029       $+139  [44.6%, 56.7%] ⚠️
  Fold 3 Dec 2025 – Apr 2026   105    63.8%     53.7%  +0.101  +0.090     $+1,962  [54.3%, 73.3%] ✅
  ────────────────────── ─────  ───────  ────────  ─────────  ─────────  ──────────  ────────────────────
  TOTAL YES UNC            559    56.9%     50.5%  +0.064  +0.054  $+11,016  [52.8%, 61.0%] ✅

────────────────────────────────────────────────────────────────────────────
  NO-ONLY UNCERTAIN BAND (entry 30–70%, outcome=NO)  — shown for contrast
─────────────────────────

In [33]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL H — Seasonal Monthly Breakdown + Fold 2 Title Inspector
# 1. Groups YES uncertain trades by calendar month → identifies safe windows
# 2. Samples Fold 2 (Aug–Dec 2025) market titles → confirms seasonal hypothesis
# 3. Produces a four-season trading calendar (TRADE / CAUTION / AVOID)
# ═══════════════════════════════════════════════════════════════════════════
import random as _rng_h
from datetime import datetime, timezone
from collections import defaultdict

if "lt_all_ev" not in globals() or not lt_all_ev:
    print("❌  Run Cell D first.")
else:
    ENTRY_LOW_H  = 0.30
    ENTRY_HIGH_H = 0.70
    TAKER_FEE_H  = 0.02

    yes_unc_h = [t for t in lt_all_ev
                 if ENTRY_LOW_H <= t["price"] <= ENTRY_HIGH_H
                 and t["outcome"].lower() == "yes"]

    # ── Monthly fee-edge breakdown ─────────────────────────────────────────
    by_month = defaultdict(list)
    for t in yes_unc_h:
        mo = datetime.fromtimestamp(t["timestamp"], tz=timezone.utc).strftime("%Y-%m")
        by_month[mo].append(t)

    print("=" * 76)
    print("CELL H — SEASONAL BREAKDOWN: YES-UNCERTAIN FEE-EDGE BY CALENDAR MONTH")
    print("=" * 76)
    print(f"  {'Month':<10}  {'n':>5}  {'WR':>7}  {'Implied':>8}  {'Fee edge':>9}  {'Net PnL':>10}  Signal")
    print(f"  {'─'*10}  {'─'*5}  {'─'*7}  {'─'*8}  {'─'*9}  {'─'*10}  {'─'*10}")

    go_months    = []
    no_go_months = []

    for mo in sorted(by_month):
        trades = by_month[mo]
        n      = len(trades)
        wins   = sum(1 for t in trades if t["won"])
        wr     = wins / n
        avg_e  = sum(t["price"] for t in trades) / n
        net    = sum(t["net_pnl"] for t in trades)
        fe     = wr - avg_e - avg_e * TAKER_FEE_H
        flag   = "✅ GO   " if fe > 0.05 else ("⚠️  weak " if fe > 0 else "❌ skip  ")
        if fe > 0.05:
            go_months.append(mo)
        elif fe <= 0:
            no_go_months.append(mo)
        print(f"  {mo:<10}  {n:>5}  {wr*100:>6.1f}%  {avg_e*100:>7.1f}%  "
              f"{fe:>+.3f}  ${net:>+9,.0f}  {flag}")

    print()
    print(f"  ✅ Strong GO months  (fee_edge > +5%): {', '.join(go_months)  or 'none'}")
    print(f"  ❌ Skip months       (fee_edge ≤  0%): {', '.join(no_go_months) or 'none'}")

    # ── Fold 2 title sample ────────────────────────────────────────────────
    # Rebuild fold boundaries from Cell D/G parameters
    now_ts_h    = int(datetime.now(timezone.utc).timestamp())
    months_h    = LT_MONTHS_BACK if "LT_MONTHS_BACK" in globals() else 12
    start_ts_h  = now_ts_h - months_h * 30 * 24 * 3600
    fold_size_h = (now_ts_h - start_ts_h) // 3
    start_f2_h  = start_ts_h + fold_size_h
    end_f2_h    = start_f2_h + fold_size_h

    fold2_yes  = [t for t in yes_unc_h if start_f2_h <= t["timestamp"] < end_f2_h]
    fold2_won  = [t for t in fold2_yes if t["won"]]
    fold2_lost = [t for t in fold2_yes if not t["won"]]

    print()
    print("─" * 76)
    f2_lo = datetime.fromtimestamp(start_f2_h, tz=timezone.utc).strftime("%b %Y")
    f2_hi = datetime.fromtimestamp(end_f2_h,   tz=timezone.utc).strftime("%b %Y")
    print(f"  FOLD 2 ({f2_lo} – {f2_hi}): SAMPLE MARKET TITLES")
    print("─" * 76)
    print(f"  Total YES uncertain trades: {len(fold2_yes)}   "
          f"won={len(fold2_won)}  lost={len(fold2_lost)}")
    print()

    sample_won  = _rng_h.sample(fold2_won,  min(12, len(fold2_won)))
    sample_lost = _rng_h.sample(fold2_lost, min(6,  len(fold2_lost)))

    print("  ── 12 WON trades (sample) ──")
    for t in sorted(sample_won, key=lambda x: x["timestamp"]):
        mo = datetime.fromtimestamp(t["timestamp"], tz=timezone.utc).strftime("%b %Y")
        print(f"    [{mo}] entry={t['price']:.2f}  {t['title'][:68]}")

    print(f"\n  ── 6 LOST trades (sample) ──")
    for t in sorted(sample_lost, key=lambda x: x["timestamp"]):
        mo = datetime.fromtimestamp(t["timestamp"], tz=timezone.utc).strftime("%b %Y")
        print(f"    [{mo}] entry={t['price']:.2f}  {t['title'][:68]}")

    # ── Four-season trading calendar ──────────────────────────────────────
    print()
    print("─" * 76)
    print("  FOUR-SEASON TRADING CALENDAR (based on 12-month data)")
    print("─" * 76)
    seasons = {
        "Spring  (Mar–May)": ["03", "04", "05"],
        "Summer  (Jun–Aug)": ["06", "07", "08"],
        "Fall    (Sep–Nov)": ["09", "10", "11"],
        "Winter  (Dec–Feb)": ["12", "01", "02"],
    }
    for season, months_s in seasons.items():
        season_trades = [
            t for t in yes_unc_h
            if datetime.fromtimestamp(t["timestamp"], tz=timezone.utc).strftime("%m") in months_s
        ]
        if not season_trades:
            print(f"  {season:<22}  (no data)")
            continue
        n     = len(season_trades)
        wins  = sum(1 for t in season_trades if t["won"])
        wr    = wins / n
        avg_e = sum(t["price"] for t in season_trades) / n
        fe    = wr - avg_e - avg_e * TAKER_FEE_H
        net   = sum(t["net_pnl"] for t in season_trades)
        rec   = "✅ TRADE  " if fe > 0.03 else ("⚠️  CAUTION" if fe > 0 else "❌ AVOID  ")
        print(f"  {season:<22}  n={n:>3}  wr={wr*100:>5.1f}%  implied={avg_e*100:>5.1f}%  "
              f"fee_edge={fe:>+.3f}  net=${net:>+7,.0f}  → {rec}")

    print()
    print("=" * 76)
    print("  HYPOTHESIS CHECK:")
    print("  If Fold 2 WON titles are mostly 'hurricane / extreme-heat / flood'")
    print("  keywords → the YES-uncertain edge IS seasonal (Aug–Nov storm season).")
    print("  If they are mixed → the edge is driven by wallet quality, not season.")
    print("  → Use Cell I (accuracy gate) to test the wallet-quality hypothesis.")


CELL H — SEASONAL BREAKDOWN: YES-UNCERTAIN FEE-EDGE BY CALENDAR MONTH
  Month           n       WR   Implied   Fee edge     Net PnL  Signal
  ──────────  ─────  ───────  ────────  ─────────  ──────────  ──────────
  2025-04        50    52.0%     45.6%  +0.055  $     +239  ✅ GO   
  2025-05        56    82.1%     46.9%  +0.343  $   +2,216  ✅ GO   
  2025-06        42    76.2%     50.8%  +0.244  $     +925  ✅ GO   
  2025-07        59    50.8%     43.2%  +0.068  $   +6,022  ✅ GO   
  2025-08        56    16.1%     54.6%  -0.396  $     -942  ❌ skip  
  2025-09        24    62.5%     53.5%  +0.080  $     -412  ✅ GO   
  2025-10        57    35.1%     46.6%  -0.125  $     -488  ❌ skip  
  2025-11        58    69.0%     54.3%  +0.136  $     +937  ✅ GO   
  2025-12       148    64.2%     54.0%  +0.091  $   +2,569  ✅ GO   
  2026-01         9    55.6%     54.3%  +0.001  $      -50  ⚠️  weak 

  ✅ Strong GO months  (fee_edge > +5%): 2025-04, 2025-05, 2025-06, 2025-07, 2025-09, 2025-11, 2025-12

In [34]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL I — Accuracy-Gated YES Signal Filter
# Filters YES uncertain trades to only those from wallets with a proven
# historical win rate (≥55% over ≥20 resolved bets in this dataset).
# Tests whether gating on wallet quality:
#   (a) improves the aggregate fee-edge
#   (b) repairs the failing walk-forward folds
#   (c) identifies a small set of consistently reliable wallets to copy
# ═══════════════════════════════════════════════════════════════════════════
import random as _rng_i
from datetime import datetime, timezone
from collections import defaultdict

if "lt_all_ev" not in globals() or not lt_all_ev:
    print("❌  Run Cell D first.")
else:
    # ── Gate parameters (tune these) ──────────────────────────────────────
    MIN_BETS_I  = 20    # minimum resolved bets to trust a wallet
    MIN_WR_I    = 0.55  # minimum win rate threshold
    ENTRY_LOW_I = 0.30
    ENTRY_HIGH_I= 0.70
    TAKER_FEE_I = 0.02

    # ── Build per-wallet stats from ALL lt_all_ev ─────────────────────────
    wallet_stats = defaultdict(lambda: {"wins": 0, "total": 0})
    for t in lt_all_ev:
        w = t.get("wallet", "")
        if not w:
            continue
        wallet_stats[w]["total"] += 1
        if t["won"]:
            wallet_stats[w]["wins"] += 1

    trusted_wallets = {
        w for w, s in wallet_stats.items()
        if s["total"] >= MIN_BETS_I and s["wins"] / s["total"] >= MIN_WR_I
    }

    print("=" * 76)
    print("CELL I — ACCURACY-GATED YES UNCERTAIN FILTER")
    print("=" * 76)
    print(f"  Gate: wallet WR ≥ {MIN_WR_I*100:.0f}%  over ≥ {MIN_BETS_I} resolved bets")
    print(f"  Unique wallets in dataset  : {len(wallet_stats):,}")
    print(f"  Wallets passing gate       : {len(trusted_wallets):,}  "
          f"({len(trusted_wallets)/max(1,len(wallet_stats))*100:.1f}% of wallets)")
    print()

    # ── YES uncertain — unfiltered vs gated ──────────────────────────────
    yes_unc_i     = [t for t in lt_all_ev
                     if ENTRY_LOW_I <= t["price"] <= ENTRY_HIGH_I
                     and t["outcome"].lower() == "yes"]
    yes_unc_gated = [t for t in yes_unc_i
                     if t.get("wallet", "") in trusted_wallets]

    def _show_i(label, trades):
        if not trades:
            print(f"  {label}: no trades")
            return {}
        n     = len(trades)
        wins  = sum(1 for t in trades if t["won"])
        wr    = wins / n
        avg_e = sum(t["price"] for t in trades) / n
        net   = sum(t["net_pnl"] for t in trades)
        fe    = wr - avg_e - avg_e * TAKER_FEE_I
        print(f"  {label:<42}  n={n:>4}  wr={wr*100:>5.1f}%  "
              f"avg_e={avg_e:.3f}  fee_edge={fe:>+.3f}  net=${net:>+9,.0f}")
        return {"n": n, "wr": wr, "avg_e": avg_e, "fe": fe, "net": net}

    s_all   = _show_i("YES uncertain — ALL wallets", yes_unc_i)
    s_gated = _show_i(
        f"YES uncertain — trusted (WR≥{MIN_WR_I:.0%}, n≥{MIN_BETS_I})",
        yes_unc_gated
    )

    # ── Gated walk-forward ────────────────────────────────────────────────
    now_ts_i    = int(datetime.now(timezone.utc).timestamp())
    months_i    = LT_MONTHS_BACK if "LT_MONTHS_BACK" in globals() else 12
    start_ts_i  = now_ts_i - months_i * 30 * 24 * 3600
    fold_size_i = (now_ts_i - start_ts_i) // 3

    def _fold_label_i(k):
        lo = datetime.fromtimestamp(start_ts_i + k * fold_size_i, tz=timezone.utc)
        hi = datetime.fromtimestamp(start_ts_i + (k+1) * fold_size_i, tz=timezone.utc)
        return f"{lo.strftime('%b %Y')}–{hi.strftime('%b %Y')}"

    def _fe_i(trades):
        if not trades:
            return float("nan"), float("nan"), 0
        n = len(trades)
        wins = sum(1 for t in trades if t["won"])
        avg_e = sum(t["price"] for t in trades) / n
        return wins/n - avg_e - avg_e * TAKER_FEE_I, sum(t["net_pnl"] for t in trades), n

    print()
    print("─" * 76)
    print("  GATED WALK-FORWARD (3 folds)")
    print("─" * 76)
    print(f"  {'Fold':<26}  {'n (all)':>8}  {'n (gated)':>10}  "
          f"{'fe (all)':>10}  {'fe (gated)':>11}  {'Δ':>7}  Result")
    print(f"  {'─'*26}  {'─'*8}  {'─'*10}  {'─'*10}  {'─'*11}  {'─'*7}  {'─'*6}")

    gated_pos = 0
    for k in range(3):
        lo = start_ts_i + k * fold_size_i
        hi = lo + fold_size_i
        fall   = [t for t in yes_unc_i     if lo <= t["timestamp"] < hi]
        fgated = [t for t in yes_unc_gated if lo <= t["timestamp"] < hi]

        fe_all,   _, n_all   = _fe_i(fall)
        fe_gated, _, n_gated = _fe_i(fgated)
        delta = (fe_gated - fe_all) if (fe_all == fe_all and fe_gated == fe_gated) else float("nan")

        fe_a_s = f"{fe_all:>+.3f}"   if fe_all   == fe_all   else "   N/A"
        fe_g_s = f"{fe_gated:>+.3f}" if fe_gated == fe_gated else "   N/A"
        d_s    = f"{delta:>+.3f}"    if delta    == delta    else "   N/A"
        flag   = "✅" if (fe_gated == fe_gated and fe_gated > 0) else "❌"
        if fe_gated == fe_gated and fe_gated > 0:
            gated_pos += 1

        print(f"  Fold {k+1} {_fold_label_i(k):<20}  {n_all:>8}  {n_gated:>10}  "
              f"{fe_a_s:>10}  {fe_g_s:>11}  {d_s:>7}  {flag}")

    # ── Sensitivity sweep ─────────────────────────────────────────────────
    print()
    print("─" * 76)
    print("  SENSITIVITY: different gate thresholds")
    print("─" * 76)
    print(f"  {'MIN_WR × MIN_BETS':<22}  {'n trades':>9}  {'WR':>7}  {'fee_edge':>9}  {'net PnL':>10}")
    print(f"  {'─'*22}  {'─'*9}  {'─'*7}  {'─'*9}  {'─'*10}")
    for min_wr in (0.50, 0.55, 0.60, 0.65):
        for min_n in (10, 20, 30):
            tw = {w for w, s in wallet_stats.items()
                  if s["total"] >= min_n and s["wins"]/s["total"] >= min_wr}
            g = [t for t in yes_unc_i if t.get("wallet","") in tw]
            if not g:
                continue
            n    = len(g); wins = sum(1 for t in g if t["won"]); wr = wins/n
            avg_e= sum(t["price"] for t in g)/n; net = sum(t["net_pnl"] for t in g)
            fe   = wr - avg_e - avg_e * TAKER_FEE_I
            tag  = "◀ current" if (min_wr == MIN_WR_I and min_n == MIN_BETS_I) else ""
            print(f"  WR≥{min_wr:.0%} & n≥{min_n:<3} ({len(tw):>3} wallets)  "
                  f"{n:>9}  {wr*100:>6.1f}%  {fe:>+.3f}  ${net:>+9,.0f}  {tag}")

    # ── Top trusted wallets ───────────────────────────────────────────────
    print()
    print("─" * 76)
    print("  TOP 10 TRUSTED WALLETS (by trade count in YES uncertain band)")
    print("─" * 76)
    if yes_unc_gated:
        by_wallet_i = defaultdict(list)
        for t in yes_unc_gated:
            by_wallet_i[t.get("wallet", "")].append(t)
        top10 = sorted(by_wallet_i.items(), key=lambda x: -len(x[1]))[:10]
        print(f"  {'Wallet':<20}  {'n':>5}  {'WR':>7}  {'fee_edge':>9}  {'net_pnl':>10}  "
              f"{'all-time WR':>12}  {'all-time n':>11}")
        print(f"  {'─'*20}  {'─'*5}  {'─'*7}  {'─'*9}  {'─'*10}  {'─'*12}  {'─'*11}")
        for wallet, trades in top10:
            n     = len(trades)
            wins  = sum(1 for t in trades if t["won"])
            wr    = wins / n
            avg_e = sum(t["price"] for t in trades) / n
            fe    = wr - avg_e - avg_e * TAKER_FEE_I
            net   = sum(t["net_pnl"] for t in trades)
            all_s = wallet_stats[wallet]
            all_wr = all_s["wins"] / all_s["total"] if all_s["total"] else 0
            print(f"  {wallet[:20]:<20}  {n:>5}  {wr*100:>6.1f}%  {fe:>+.3f}  "
                  f"${net:>+9,.0f}  {all_wr*100:>11.1f}%  {all_s['total']:>11}")
    else:
        print("  (no trusted wallets found — try lowering the thresholds)")

    # ── Final verdict ─────────────────────────────────────────────────────
    print()
    print("=" * 76)
    print("VERDICT")
    print("=" * 76)
    n_all_t   = len(yes_unc_i)
    n_gated_t = len(yes_unc_gated)
    print(f"  Gate reduces trade count: {n_all_t} → {n_gated_t} "
          f"({n_all_t - n_gated_t} filtered, {(n_all_t - n_gated_t)/max(1,n_all_t)*100:.0f}%)")
    fe_all_t   = s_all.get("fe",   float("nan"))
    fe_gated_t = s_gated.get("fe", float("nan"))
    if fe_all_t == fe_all_t:
        print(f"  Fee-edge change: {fe_all_t:>+.3f} → {fe_gated_t:>+.3f}  "
              f"(Δ = {fe_gated_t - fe_all_t:>+.3f})")
    print(f"  Gated folds with positive fee-edge: {gated_pos}/3")
    print()
    if gated_pos == 3:
        print("  ✅ Accuracy gate is HIGHLY effective — positive edge in all 3 folds.")
        print(f"     → Use MIN_WR={MIN_WR_I:.0%}, MIN_BETS={MIN_BETS_I} as production filter.")
        print("     → Monitor the top trusted wallets in weather_whale_monitor.py.")
    elif gated_pos == 2:
        print("  ✅ Accuracy gate HELPS — consistent in 2/3 folds.")
        print("     → Paper-trade gated YES uncertain signals immediately.")
        print("     → Combine with seasonal filter (Cell H) to avoid the weak fold.")
    elif gated_pos == 1:
        print("  ⚠️  Gate helps marginally. The underlying problem is seasonal, not wallet quality.")
        print("     → Try combining Cell H seasonal calendar with this gate.")
        print("     → Or increase MIN_BETS to 30 for a stricter quality filter.")
    else:
        print("  ❌ Accuracy gate does NOT help. Edge issue is regime/seasonal, not wallet quality.")
        print("     → Focus strategy on seasonal calendar from Cell H.")
        print("     → Do not copy-trade whale wallets blindly across all seasons.")
    print()
    print("  NEXT: Run Cell D with LT_SAMPLE=300 to increase sample size,")
    print("  then re-run Cells E, G, H, I for statistically robust conclusions.")


CELL I — ACCURACY-GATED YES UNCERTAIN FILTER
  Gate: wallet WR ≥ 55%  over ≥ 20 resolved bets
  Unique wallets in dataset  : 2,905
  Wallets passing gate       : 59  (2.0% of wallets)

  YES uncertain — ALL wallets                 n= 559  wr= 56.9%  avg_e=0.505  fee_edge=+0.054  net=$  +11,016
  YES uncertain — trusted (WR≥55%, n≥20)      n=  76  wr= 67.1%  avg_e=0.537  fee_edge=+0.123  net=$   +1,034

────────────────────────────────────────────────────────────────────────────
  GATED WALK-FORWARD (3 folds)
────────────────────────────────────────────────────────────────────────────
  Fold                         n (all)   n (gated)    fe (all)   fe (gated)        Δ  Result
  ──────────────────────────  ────────  ──────────  ──────────  ───────────  ───────  ──────
  Fold 1 Apr 2025–Aug 2025          223          39      +0.123       +0.240   +0.117  ✅
  Fold 2 Aug 2025–Dec 2025          231          31      -0.029       -0.015   +0.014  ❌
  Fold 3 Dec 2025–Apr 2026          105      

## 📊 Current Results — Interpretation & Next Steps

### 1. What the 12-Month Backtest Tells Us

| Bucket | Trades | Win Rate | Avg Entry (Implied) | Fee-Adj Edge | Net PnL |
|---|---|---|---|---|---|
| All trades (Cell D) | ~9,205 | ~56–57% | ~50% | **+1.5%** | +$13 K |
| Uncertain band 30–70% | ~1,120 | ~56.7% | ~50% | **+1.5%** | +$13 K |
| YES uncertain | subset | higher | ~50% | varies by fold | see G |
| NO uncertain | subset | lower | ~50% | mostly negative | see G |

**Bootstrap 95% CI** on the uncertain band straddles the implied probability →  
**⚠️ edge is borderline — statistically inconclusive with the current sample size.**

---

### 2. Walk-Forward Validation (Cell E) — The Most Important Signal

| Fold | Period | Unc Trades | WR | Fee-Adj Edge | Result |
|---|---|---|---|---|---|
| Fold 1 | Apr–Aug 2025 | 436 | 63.5% | **+8.4%** | ✅ Strong |
| Fold 2 | Aug–Dec 2025 | 466 | 55.4% | **−0.2%** | ❌ Near zero |
| Fold 3 | Dec 2025–Apr 2026 | 218 | 45.9% | **−8.6%** | ❌ Negative |

**Key insight:** The edge is strongly seasonal. It exists in spring (Fold 1) but vanishes in late autumn and reverses in winter.  
This rules out a universal copy-trade strategy — a **seasonal filter is mandatory**.

---

### 3. YES-Only Uncertain Band (Cell G)

- YES bets in the 30–70% band drive the aggregate positive edge.
- NO bets in the same band show a flat or negative edge across folds.
- Fold 1 is the dominant driver; the total CI likely straddles the implied probability.
- **Actionable:** Trade YES uncertain only, spring/summer months, with a quality gate.

---

### 4. Seasonal Calendar (Cell H)

| Season | Fee-Edge Signal |
|---|---|
| Spring Mar–May | ✅ GO — strongest edge |
| Summer Jun–Aug | ✅ GO — moderate edge |
| Fall Sep–Nov | ⚠️ Weak — hurricane resolution noise |
| Winter Dec–Feb | ❌ AVOID — edge reverses |

---

### 5. Accuracy Gate (Cell I)

Gating to wallets with **WR ≥ 55% over ≥ 20 resolved bets** filters the wallet universe significantly.  
Key check: if `gated_pos ≥ 2/3`, the wallet-quality hypothesis is validated.  
If `gated_pos = 1/3`, the problem is seasonal (not wallet quality) — apply Cell H calendar instead.

---

### 6. Live Whale Monitor (Cell from live data)

Recent top-wallet activity (last week):
- All 4 recent trades are **NO bets at 0.91–0.99 price** (very high confidence markets).
- These are **outside the uncertainty band** — not tradeable with our filter.
- Signal interpretation: top weather traders believe 2026 will **not** be a record-hot year and **not** hit specific daily temps.
- No consensus burst detected (single-wallet trades).

---

### 7. Critical Bugs Found

| Priority | Bug | Impact |
|---|---|---|
| 🔴 CRITICAL | Arb scanner sums mid-prices (always ≈1.0) | Zero arb alerts ever correct |
| 🔴 CRITICAL | No fee model in original backtest → edge overstated | Requires 2% fee deduction |
| 🟡 MEDIUM | `seen_hashes` resets on restart → duplicate DB alerts | Stale data pollution |
| 🟡 MEDIUM | Consensus burst checks only current batch, not accumulated | Misses multi-loop bursts |
| 🟢 MINOR | Bootstrap CI upper index off-by-one (fixed in Cell D & G) | Negligible CI shift |

---

### 8. Next Steps — Prioritised

**Do immediately:**
1. ✅ Fix BUG-1 in `bot.py`: use `yes_ask` / `no_ask` in `check_arbitrage()`, not `yes_price` / `no_price`.
2. ✅ Fix BUG-2: deduct 2% taker fee in `strategy_engine.EdgeIndicator`.
3. 📌 Re-run Cell D with `LT_SAMPLE = 500` to tighten the bootstrap CI.

**This week:**
4. Apply seasonal filter: only enter YES uncertain trades in Mar–Aug.
5. Apply accuracy gate: wallets with WR ≥ 55% over ≥ 20 resolved bets (`MIN_WR_I=0.55, MIN_BETS_I=20`).
6. Fix BUG-4: seed `seen_hashes` from existing DB `transaction_hash` on `weather_whale_monitor` restart.
7. Start `weather_snapshot_daemon.py` in background for rank-velocity data.

**Next week:**
8. Paper-trade: YES uncertain + consensus + seasonal filter + accuracy gate combo.
9. Track hypothetical PnL for 4–8 weeks.
10. Run walk-forward over 24 months once more resolved markets accumulate.

**Later (live trading):**
11. Kelly-sized positions once paper-trading confirms 3+ months of positive edge.

---

### 9. The One Strategy Worth Paper-Trading Now

> **Filter:** YES uncertain (entry 30–70%) + CONSENSUS (≥2 wallets) + trusted wallet (WR ≥ 55%, n ≥ 20) + season Mar–Aug.  
> **Estimated universe:** ~53 consensus trades over 12 months → ~4–5/month.  
> **Expected fee-adjusted edge:** +5–8% (Fold 1 baseline), subject to seasonal degradation in winter.  
> **Risk:** Edge may not persist — **paper-trade first, do not go live before 3 months of validation.**


## 📖 Backtest Guide — Step-by-Step How-To

This section explains **exactly** how the Polymarket weather backtest is constructed, what records to retrieve, how to compare them, which backtest framework to use, which signals to trade, and defines all key terminology.

---

### 🔑 Key Definitions

#### Wallet-Centric Strategy
A strategy that **starts from a fixed list of wallets** (traders) and follows every trade they make.  
- **Pro:** Built-in quality signal — the wallet already has a proven record on the leaderboard.  
- **Con:** The sample is biased toward the wallets you selected; if they become infrequent traders, signal dries up.  
- **Example (Cell backtest Part A):** Follow the top-20 wallets from the Polymarket weather leaderboard, copy every BUY trade they make on a weather market within the uncertainty band.

#### Market-Centric Strategy
A strategy that **starts from a set of markets** and analyzes every trader that touched those markets.  
- **Pro:** Unbiased view of the market; captures all liquidity, not just known whale wallets.  
- **Con:** No quality filter on wallets — you evaluate all traders including uninformed ones, which dilutes the edge.  
- **Example (Cell backtest Part B):** Sample 50–300 resolved weather markets, pull every BUY trade ≥ $10 notional, evaluate them against the known resolution outcome.

#### Uncertainty Band
The price range **[0.30, 0.70]** for a binary outcome (YES/NO market).  
- At 0.30 price, the market says there is a 30% chance of YES; at 0.70, a 70% chance.  
- Trades inside this band are placed when the market is **genuinely uncertain** — neither side is heavily favored.  
- **Why it matters:** Trades at 0.05 (very unlikely) or 0.95 (near-certainty) give small payoffs or small losses but enormous variance. Trades in [0.30, 0.70] have the best risk-adjusted edge potential.  
- **Complement:** Trades *outside* this band (< 0.30 or > 0.70) are high-confidence bets — they are included in the backtest to understand the full picture but are not the focus of the strategy.

#### Fee-Adjusted Edge
$$\text{fee\_edge} = \text{Win Rate} - \bar{p} - \bar{p} \times f$$
where $\bar{p}$ is the average entry price (the market's implied probability) and $f = 0.02$ (2% Polymarket taker fee).  
- **Positive fee_edge** means the strategy earns more than the market implied AND covers the fee.  
- **Negative fee_edge** means the strategy loses to the market after fees — do not trade.

#### Walk-Forward Validation
Split the historical data into **chronological folds** (no shuffling). Train on earlier folds, validate on later folds.  
- Used here as a 3-fold chronological split (not a "training" split, but a stability check).
- A strategy that shows positive fee_edge in all 3 folds is **robust**; only in 1/3 folds means the edge is seasonal or noisy.

#### Bootstrap 95% Confidence Interval (CI)
Re-sample the win/loss results 2,000+ times (with replacement) and compute the win rate each time.  
Sort results → take the 2.5th and 97.5th percentiles as the CI bounds.  
- If the **lower bound > implied probability**, the edge is statistically significant.  
- If the CI **straddles** the implied probability, more data is needed.

#### Consensus Signal
Two or more **different wallets** BUY the same (market, outcome) within a 24-hour window.  
- Consensus signals are stronger than a single-wallet whale trade because multiple independent actors agree.  
- In the 3-month backtest: 53 consensus signals, 54.7% WR vs 47% implied → +7.7% gross edge.

#### Accuracy Gate
A pre-filter applied to wallets: only use signals from wallets whose **historical win rate ≥ 55% over ≥ 20 resolved bets**.  
- Removes noise from lucky/uninformed wallets.  
- Cell I tests whether this gate improves the fee-adjusted edge across all 3 walk-forward folds.

---

### 📋 Step-by-Step Backtest Process

#### STEP 1 — Build the Market Universe

**What to retrieve:**  
Use the Gamma API to get all resolved weather markets:

```
GET https://gamma-api.polymarket.com/markets
  ?closed=true
  &archived=true
  &tag_slug=weather      ← or filter by keyword after fetching
```

**Fields you need:**  
- `conditionId` — unique market identifier (used to query trades)  
- `resolvedOutcome` — "Yes" or "No" (the ground truth for evaluating bets)  
- `question` — human-readable title  
- `volumeNum` — total trading volume (filter out low-liquidity markets)

**Filter criteria:**  
- Keep markets with keyword match: "hurricane", "temperature", "storm", "earthquake", "hottest", "rainfall", "tornado", "celsius", "fahrenheit", "climate", etc.  
- Discard markets resolved > 18 months ago (data quality degrades).  
- Discard markets with volume < $5,000 (thin liquidity → few trades to analyze).

**In code (Cell D):**
```python
fetcher = WeatherMarketFetcher(page_size=50, max_pages=20)
closed_mkts = fetcher.fetch_closed(max_pages=20)
resolved_map = {m["conditionId"]: m["resolvedOutcome"] for m in closed_mkts
                if m.get("conditionId") and m.get("resolvedOutcome")}
```

---

#### STEP 2 — Retrieve Trade Records

**For Wallet-Centric (Part A):**  
First get the leaderboard:
```
GET https://data-api.polymarket.com/leaderboard
  ?spotlightType=weather
  &limit=20
```
Then for each wallet:
```
GET https://data-api.polymarket.com/trades
  ?user={proxyWallet}
  &takerOnly=true        ← BUG-6: also try false to catch maker trades
  &limit=100
  &offset={page * 100}
```

**For Market-Centric (Part B / Cell D):**  
For each sampled conditionId:
```
GET https://data-api.polymarket.com/trades
  ?market={conditionId}
  &takerOnly=true
  &limit=100
  &offset={page * 100}
```

**Fields you need per trade:**  
| Field | Description |
|---|---|
| `outcome` | "Yes" or "No" (what the trader bet on) |
| `price` | Entry price (0–1), equals the implied probability |
| `size` | Number of shares purchased |
| `proxyWallet` | Trader's wallet address |
| `timestamp` | Unix timestamp of the trade |
| `side` | "BUY" or "SELL" — filter to BUY only |
| `title` | Market description (for inspection) |

**Filters to apply:**  
- `side == "BUY"` (selling is a different strategy — out of scope)  
- `notional = price × size >= $10` (ignore micro-trades)  
- `timestamp >= cutoff` (stay within the lookback window)

---

#### STEP 3 — Evaluate Each Trade

Compare each trade's `outcome` against the market's `resolvedOutcome`:

```python
won   = trade["outcome"].lower() == resolved_outcome.lower()
gross = (1.0 - entry) * size  if won  else  -entry * size
fee   = entry * size * 0.02   # 2% taker fee on notional
net   = gross - fee
```

**Metrics to compute per trade:**  
- `won` (bool): True if the bet was correct  
- `gross_pnl`: Raw profit before fees  
- `net_pnl`: Profit after the 2% taker fee  
- `fee_edge`: `win_rate - avg_entry - avg_entry × 0.02` (aggregate metric)

---

#### STEP 4 — Slice into Comparison Buckets

| Bucket | Filter | Purpose |
|---|---|---|
| All trades | none | Baseline — expected ≈ 0 edge |
| Uncertainty band | `0.30 ≤ price ≤ 0.70` | Where alpha is theoretically available |
| YES uncertain | band + `outcome == "yes"` | Tests YES-side bias |
| NO uncertain | band + `outcome == "no"` | Contrast / sanity check |
| CONSENSUS only | ≥2 wallets same (market, outcome) within 24h | Strongest signal bucket |
| Accuracy-gated | wallet WR ≥ 55% over ≥ 20 resolved bets | Quality-filtered subset |

Compare each bucket's `fee_edge` against zero:  
- `fee_edge > 0` → strategy earns alpha after fees  
- `fee_edge < 0` → do not trade

---

#### STEP 5 — Which Backtest Framework to Use

We do **not** use a third-party backtesting library (Backtrader, Zipline, etc.) because:
1. Polymarket outcomes are binary — no continuous price series to simulate.
2. The "position" is just a single BUY bet that resolves to WIN or LOSS.
3. Each trade is independent — no portfolio management or position sizing needed yet.

**Framework used: pure Python with bootstrap CI**  
```
Data → Filter → Evaluate PnL → Aggregate stats → Bootstrap CI → Walk-forward split
```

**Walk-Forward Setup (Cell E):**
```python
span      = now_ts - start_ts          # e.g. 12 months in seconds
fold_size = span // 3                  # 3 equal folds ≈ 4 months each
fold_k    = [t for t in data if (start_ts + k*fold_size) <= t["timestamp"]
                                      < (start_ts + (k+1)*fold_size)]
```
Compute `fee_edge` per fold; require ≥ 2/3 folds positive before trading live.

---

#### STEP 6 — Which Signals to Use

**Priority order (best to worst):**

1. **CONSENSUS + YES + Uncertainty Band + Spring/Summer + Accuracy Gate**  
   → Highest conviction, ~4–5 trades/month  
   → Expected fee_edge: +5–8% (based on Fold 1 data)

2. **WHALE + YES + Uncertainty Band + Spring/Summer + Accuracy Gate**  
   → Single large wallet, higher frequency but noisier  
   → Expected fee_edge: +2–4% (borderline, needs more data)

3. **All YES uncertain + Spring/Summer only**  
   → Broad entry, lower quality filter  
   → Expected fee_edge: +1.5–3% (marginal, monitor carefully)

4. **All trades (any confidence level)**  
   → Fee_edge ≈ 0 after 2% fee — **do not trade**.

**Signal generation logic:**
```python
# WHALE signal
if notional >= 100 and side == "BUY":
    emit whale_signal(cid, outcome, price, wallet)

# CONSENSUS signal
for (cid, outcome), sigs in group_by(cid, outcome):
    in_24h = [s for s in sigs if within_24h(anchor, s)]
    if len({s["wallet"] for s in in_24h}) >= 2:
        emit consensus_signal(cid, outcome, avg_price)
```

---

#### STEP 7 — Validate Statistical Significance

Before going live, confirm:

| Check | Threshold | Status |
|---|---|---|
| Bootstrap CI lower bound > implied prob | CI_lo > avg_entry | ⚠️ Borderline (needs 500+ trades) |
| Walk-forward folds with positive edge | ≥ 2 out of 3 | ⚠️ Currently 1/3 total, ✅ Fold 1 |
| Minimum sample size in uncertainty band | ≥ 300 trades | ✅ 1,120 trades (Cell D) |
| Seasonal filter applied | Mar–Aug only | ⬜ Not yet enforced in live bot |
| Accuracy gate applied | WR ≥ 55%, n ≥ 20 | ⬜ Cell I confirmed; not in bot yet |

---

#### STEP 8 — Paper-Trade Before Going Live

1. Enable `ENABLE_STRATEGY_PIPELINE=true` in `.env`.
2. Wire `strategy_engine.py` output to `data_logger.py`.
3. Run `weather_whale_monitor.py` continuously.
4. For each consensus signal in the uncertainty band (spring/summer): log the hypothetical trade.
5. Check weekly: is the running fee_edge still positive?
6. Go live only after **3+ months with positive fee_edge** and **CI_lo > implied probability**.

---

### 🔗 API Quick Reference

| Endpoint | Purpose |
|---|---|
| `gamma-api.polymarket.com/markets?closed=true` | Resolved market list |
| `gamma-api.polymarket.com/events?active=true` | Active events (for pending markets) |
| `data-api.polymarket.com/trades?market={cid}` | All trades on a market |
| `data-api.polymarket.com/trades?user={wallet}` | All trades by a wallet |
| `data-api.polymarket.com/leaderboard?spotlightType=weather` | Weather leaderboard |
| `clob.polymarket.com/markets/{tokenId}` | Current orderbook (ask prices for arb check) |


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL F — Next-Move Checklist & Bug Fix Demos
# Summarises the 10 bugs found and shows quick fixes that can be applied
# without breaking existing code.
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 72)
print("BUG INVENTORY & QUICK-FIX GUIDE")
print("=" * 72)

bugs = [
    ("🔴 BUG-1", "CRITICAL",
     "check_arbitrage() uses mid-prices (always sum to ~1.0).",
     "Pass yes_ask/no_ask instead of yes_price/no_price in bot.py:scan_market()"),
    ("🔴 BUG-2", "CRITICAL",
     "No 2% taker fee in profit model → edge is overstated by ~2–4%.",
     "Subtract entry*size*0.02 from every trade PnL; raise MIN_PROFIT_MARGIN to 0.05"),
    ("🔴 BUG-3", "CRITICAL",
     "Part-B uncertain bets (entry<0.70) have NEGATIVE fee-adjusted edge (−5.7%).",
     "Do not copy uncertain weather bets blindly; apply accuracy-score gating first"),
    ("🟡 BUG-4", "MEDIUM",
     "seen_hashes resets on restart → duplicate alerts in DB.",
     "Seed seen_hashes from existing whale_alerts.transaction_hash on startup"),
    ("🟡 BUG-5", "MEDIUM",
     "Consensus burst only checks within current loop batch, misses cross-loop bursts.",
     "Pass accumulated seen_alerts to detect_consensus_burst(), not just current batch"),
    ("🟡 BUG-6", "MEDIUM",
     "takerOnly=true misses large maker-side positions.",
     "Use takerOnly=false and filter by side=BUY server-side"),
    ("🟡 BUG-7", "MEDIUM",
     "_determine_winner threshold 0.9 excludes some freshly-resolved markets.",
     "Lower threshold to 0.85 OR check market.resolved boolean field"),
    ("🟢 BUG-8", "MINOR",
     "LOW_CONFIDENCE_THRESHOLD=10 is statistically too low for binary outcomes.",
     "Raise to 30 in weather_accuracy.py"),
    ("🟢 BUG-9", "MINOR",
     "ENABLE_STRATEGY_PIPELINE defaults to false — strategy_engine.py is dead code.",
     "Set ENABLE_STRATEGY_PIPELINE=true in .env; wire output to data_logger"),
    ("🟢 BUG-10","MINOR",
     "data_logger.py uses datetime.now() (local time) not UTC.",
     "Replace with datetime.now(timezone.utc) for consistent SQLite queries"),
]

for code, sev, issue, fix in bugs:
    print(f"\n  {code} [{sev}]")
    print(f"    Issue : {issue}")
    print(f"    Fix   : {fix}")

print(f"\n{'═'*72}")
print("NEXT STEPS CHECKLIST")
print("═" * 72)

steps = [
    ("✅ NOW",      "Run CELL D (extended backtest) to confirm uncertainty-band edge holds over 12 months"),
    ("✅ NOW",      "Run CELL E (walk-forward) to check edge stability across time folds"),
    ("📌 TODAY",    "Fix BUG-1 in bot.py: pass yes_ask/no_ask to check_arbitrage()"),
    ("📌 TODAY",    "Fix BUG-2: add 2% fee to strategy_engine.EdgeIndicator and profit model"),
    ("📌 TODAY",    "Fix BUG-4: seed seen_hashes from DB on weather_whale_monitor restart"),
    ("📆 THIS WEEK","Fix BUG-5: cross-loop consensus detection (pass accumulated batch)"),
    ("📆 THIS WEEK","Start weather_snapshot_daemon.py in background for rank-velocity data"),
    ("📆 THIS WEEK","Run weather_accuracy.py --top-n 50, gate signals to wallets with WA≥0.55"),
    ("📆 NEXT WEEK","Paper-trade: uncertainty-band filter + accuracy-gate, track hypothetical PnL"),
    ("📆 NEXT WEEK","Build walk-forward over 24 months (when more resolved markets accumulate)"),
    ("📅 LATER",    "Live trading: Kelly-sized positions once paper-trading confirms 3+ months edge"),
]

for timing, task in steps:
    print(f"  {timing:<16} {task}")

print(f"\n{'─'*72}")
print("KEY INSIGHT FROM CURRENT DATA:")
print("  The 3-month backtest shows +2.3% raw edge (Part A) but this becomes")
print("  essentially ZERO after 2% Polymarket taker fees.")
print("  The ONLY bucket with potentially real alpha is:")
print("    → Consensus signals (≥2 wallets) at uncertain prices (30–70%)")
print("    → 53 trades, 54.7% win rate vs 47% implied = +7.7% gross, +~5% after fees")
print("  This needs 12-month validation before going live.")
